In [1]:
// =============================================================================
// BilgePump::Library
// Layer: Library
// Purpose: Pure definitions only — no part usages, no connections.
// Every component in the bilge pump system is defined here as a
// part def with typed ports and attributes. Architecture, Requirements,
// and Analysis layers import this package and never re-define types.
//
// External engineering tool context (hypothetical integrations):
// - Sensor calibration DB  → provides waterLevel range and accuracy specs
// - Simulink/Stateflow     → provides PumpController logic model
// - IEC 61850 power bus    → provides PowerSupply nominal voltage profile
// - CFD simulation tool    → provides BilgePumpA/B flow rate & efficiency curves
// - CAD (CATIA/SolidWorks) → provides physical envelope / strainer sizing
// - P&ID tool              → provides DischargeLine pipe loss coefficients
// - IEC 60945              → provides AlarmSystem activation delay standards
// - HMI design tool        → provides OperatorInterface display/override spec
// =============================================================================

package BilgePump_Library {
    // ScalarValues provides the primitive types Real, String, Boolean used by
    // the attribute/port definitions below. Required for SysML v2 kernel
    // validation and %publish (the API stores typed elements, not just text).
    private import ScalarValues::*;

    // -------------------------------------------------------------------------
    // Attribute Definitions
    // -------------------------------------------------------------------------

    // Represents a water level measurement in metres above bilge floor.
    // SOURCE: sensor datasheet / calibration lab; range typically 0.0–1.0 m
    attribute def WaterLevelAttr { attribute waterLevel : Real; }

    // Pump operational flow rate in cubic metres per second.
    // SOURCE: CFD simulation tool (pump curve export, e.g. OpenFOAM or ANSYS Fluent)
    attribute def FlowRateAttr { attribute flowRate : Real; }

    // Hydraulic efficiency factor 0.0–1.0 (dimensionless).
    // SOURCE: CFD simulation; vendor pump test certificate
    attribute def EfficiencyAttr { attribute efficiency : Real; }

    // Pipe friction / loss factor (Darcy-Weisbach λ), dimensionless.
    // SOURCE: P&ID tool; manual hydraulic calculation per ISO 4185
    attribute def PipeLossAttr { attribute pipeLossFactor : Real; }

    // Alarm activation delay in seconds.
    // SOURCE: IEC 60945 §4.3 — must be ≤ 2.0 s for Class A alarms
    attribute def AlarmDelayAttr { attribute activationDelay_s : Real; }

    // -------------------------------------------------------------------------
    // Port Definitions
    // These represent typed signal / flow interfaces between components.
    // Conjugate ports (prefixed ~) are the receiving ends of each interface.
    // -------------------------------------------------------------------------

    // --- Level signal: Sensor → Controller ---
    // Carries the raw water-level measurement as a scalar.
    // In a real vessel integration, this maps to a 4–20 mA analog loop
    // or an NMEA 2000 PGN signal from the vessel sensor bus.
    port def LevelSignalPort {
        attribute waterLevel : Real;
    }

    // --- Pump control command: Controller → Pump ---
    // Carries a start/stop command string (stub; full model would use enum def).
    // SOURCE (out): PumpController firmware model (Simulink discrete-event output)
    // DEST (in):    Pump motor drive / VFD firmware
    port def PumpControlPort {
        attribute command : String; // "START" | "STOP" | "FAULT"
    }

    // --- Electrical power: PowerSupply → Pump ---
    // Carries bus voltage and a flag indicating whether emergency bus is active.
    // SOURCE: Vessel power management system (IEC 61850 GOOSE message)
    // DEST:   Pump motor drive; also feeds into load-shedding controller
    port def PowerPort {
        attribute voltage : Real;
        attribute isEmergency : Boolean;
    }

    // --- Fluid flow: Pump → DischargeLine ---
    // Carries volumetric flow rate through the discharge piping.
    // SOURCE: pump impeller speed sensor (tachometer) or flow meter (NMEA)
    // DEST:   Discharge overboard; MARPOL environmental compliance monitor
    port def FluidFlowPort {
        attribute flowRate : Real;
    }

    // --- System status: Controller ↔ OperatorInterface ---
    // Bidirectional; status string is a simplified representation.
    // In a real system: encoded as a CAN/NMEA 2000 status word or Modbus register.
    // DEST: Voyage Data Recorder (VDR), bridge BNWAS display
    port def StatusPort {
        attribute status : String; // "NORMAL" | "PUMPING" | "FAULT" | "OVERRIDE"
    }

    // --- Alarm trigger: Controller → AlarmSystem ---
    // Boolean flag; high = alarm condition active.
    // SOURCE: PumpController threshold comparator
    // DEST:   Audible/visual alarm unit; VDR alarm log; bridge alarm panel (IEC 60945)
    port def AlarmPort {
        attribute alarmActive : Boolean;
    }

    // --- Manual override: OperatorInterface → Controller ---
    // Allows bridge crew to force-start/stop pumps regardless of automation state.
    // SOURCE: HMI panel (physical button or touchscreen)
    // DEST:   PumpController override input register
    port def OverridePort {
        attribute override : Boolean;
    }

    // =========================================================================
    // Part Definitions — Library Layer
    // Each part def declares its ports and attributes.
    // No values are assigned here; values are bound in Architecture or Analysis.
    // =========================================================================

    // -------------------------------------------------------------------------
    // BilgeWaterSensor
    // Measures water accumulation depth in the bilge.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS:
    // ← Sensor calibration certificate (accuracy class, temperature range)
    // ← Classification society approval cert (DNV/Lloyd's type approval)
    // ENGINEERING OUTPUTS:
    // → PumpController (level signal, this model)
    // → SCADA / vessel management system (OPC-UA data point)
    // → NMEA 2000 vessel data bus (PGN 127501 binary switch bank)
    part def BilgeWaterSensor {
        // Output: water level to controller
        port levelOut : LevelSignalPort;

        // Current measured water level in metres
        attribute waterLevel : Real;
        // Sensor accuracy class (e.g., 0.01 m resolution)
        attribute accuracy_m : Real;
    }

    // -------------------------------------------------------------------------
    // PumpController
    // Central control logic and diagnostics unit.
    // In practice, this is typically a PLC or dedicated bilge controller board.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS:
    // ← BilgeWaterSensor level signal (this model)
    // ← OperatorInterface override command (this model)
    // ← Simulink/Stateflow control logic model (firmware code generation)
    // ← Classification society alarm logic spec (IEC 60945, DNV Rules Pt.4 Ch.6)
    // ENGINEERING OUTPUTS:
    // → BilgePumpA / BilgePumpB start/stop commands (this model)
    // → AlarmSystem trigger (this model)
    // → OperatorInterface status (this model)
    // → CMMS fault logger (maintenance work order generation)
    // → SCADA historian (time-series event log)
    part def PumpController {
        // Input: water level from sensor
        port levelIn : ~LevelSignalPort;
        // Output: pump A start/stop command
        port pumpAControl : PumpControlPort;
        // Output: pump B start/stop command
        port pumpBControl : PumpControlPort;
        // Output: system status to operator interface
        port statusOut : StatusPort;
        // Output: alarm trigger to alarm system
        port alarmOut : AlarmPort;
        // Input: manual override from operator
        port overrideIn : ~OverridePort;

        // Time from threshold breach to pump start command (seconds)
        attribute responseTime_s : Real;
        // Water level threshold at which pumping is triggered (metres)
        attribute triggerLevel_m : Real;
        // Time from Pump A fault detection to Pump B activation command (seconds)
        // SOURCE: Simulink/Stateflow timing model SIM-CTRL-001 §3.2
        // Normal failover: 0.8 s; delayed detection: 2.7 s; no failover: 9999.0 s
        // Required by BPS-REQ-006 (FailoverSwitchTimingRequirement)
        attribute failoverTime_s : Real;
    }

    // -------------------------------------------------------------------------
    // PowerSupply
    // Dual-source supply unit: main bus + emergency/battery backup.
    // Must maintain pump operation during main bus failure per SOLAS II-1.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS:
    // ← Vessel power management system (IEC 61850 GOOSE — bus availability)
    // ← Electrical load analysis (power budget spreadsheet / ETAP model)
    // ← Emergency generator sizing calculation
    // ENGINEERING OUTPUTS:
    // → BilgePumpA / BilgePumpB power rails (this model)
    // → Load shedding controller (IEC 61850 — priority group assignment)
    // → Vessel alarm panel (power failure indication)
    part def PowerSupply {
        // Output: power to pump A
        port powerOutA : PowerPort;
        // Output: power to pump B
        port powerOutB : PowerPort;

        // Nominal DC/AC bus voltage in volts
        attribute nominalVoltage : Real;
        // True when emergency bus has taken over
        attribute redundancyActive : Boolean;
    }

    // -------------------------------------------------------------------------
    // BilgePumpA  (primary pump)
    // Centrifugal or diaphragm pump for main bilge drainage.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS:
    // ← CFD simulation tool: pump characteristic curve (H-Q curve, ANSYS or OpenFOAM)
    // ← CAD model: physical envelope / mounting interface (CATIA/SolidWorks STEP file)
    // ← Vendor test certificate: NPSH, efficiency at rated speed
    // ENGINEERING OUTPUTS:
    // → DischargeLine flow (this model)
    // → CMMS run-hour counter (predictive maintenance trigger at 250 h)
    // → SCADA: flow rate telemetry (OPC-UA node)
    part def BilgePumpA {
        // Input: power from supply
        port powerIn : ~PowerPort;
        // Input: start/stop command from controller
        port controlIn : ~PumpControlPort;
        // Output: fluid flow to discharge line
        port flowOut : FluidFlowPort;

        // Volumetric flow rate at rated speed (m³/s)
        attribute flowRate : Real;
        // Hydraulic efficiency at operating point (0.0–1.0)
        attribute efficiency : Real;
        // Cumulative run hours (sourced from CMMS at model instantiation)
        attribute runHours : Real;
    }

    // -------------------------------------------------------------------------
    // BilgePumpB  (redundant pump)
    // Identical capability to Pump A; activates on A failure or override.
    // Classification rules (DNV Pt.4 Ch.6) require independent power feed.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS: same as BilgePumpA
    // ENGINEERING OUTPUTS: same as BilgePumpA
    part def BilgePumpB {
        port powerIn : ~PowerPort;
        port controlIn : ~PumpControlPort;
        port flowOut : FluidFlowPort;

        attribute flowRate : Real;
        attribute efficiency : Real;
        attribute runHours : Real;
        // Flag confirming this unit is the designated redundant pump
        attribute isRedundant : Boolean;
    }

    // -------------------------------------------------------------------------
    // DischargeLine
    // Overboard discharge piping including strainer basket and check valves.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS:
    // ← P&ID drawing tool: pipe routing, valve schedule (SmartPlant P&ID)
    // ← CAD: strainer basket sizing, pipe bore (CATIA Piping & Tubing)
    // ← Hydraulic calculation: Darcy-Weisbach friction factor per ISO 4185
    // ENGINEERING OUTPUTS:
    // → Overboard discharge (physical)
    // → MARPOL environmental compliance monitor (discharge rate < 15 ppm oily water)
    // → Port state control inspection record (oil content meter reading)
    part def DischargeLine {
        // Input: flow from pump A
        port flowInA : ~FluidFlowPort;
        // Input: flow from pump B
        port flowInB : ~FluidFlowPort;

        // Darcy-Weisbach pipe friction/loss factor (dimensionless)
        attribute pipeLossFactor : Real;
        // Strainer free-flow area (m²); sized from CAD
        attribute strainerArea_m2 : Real;
    }

    // -------------------------------------------------------------------------
    // AlarmSystem
    // Audible and visual alarm unit installed at the pump control station
    // and mirrored to the bridge alarm panel per IEC 60945 §4.3.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS:
    // ← PumpController alarm trigger (this model)
    // ← IEC 60945 alarm standard: activation delay ≤ 2.0 s, sound level ≥ 75 dB
    // ← Classification society alarm table (DNV Rules Pt.4 Ch.6 Table 1)
    // ENGINEERING OUTPUTS:
    // → OperatorInterface notification (this model)
    // → Bridge BNWAS (bridge navigational watch alarm system)
    // → Voyage Data Recorder (VDR) alarm event log
    // → Shore-based fleet monitoring system (satellite AIS/IoT gateway)
    part def AlarmSystem {
        // Input: alarm condition from controller
        port alarmIn : ~AlarmPort;
        // Output: notification to operator interface
        port notifyOut : AlarmPort;

        // Delay from trigger to annunciation in seconds (must be ≤ 2.0)
        attribute activationDelay_s : Real;
        // True when alarm is currently sounding
        attribute isActive : Boolean;
    }

    // -------------------------------------------------------------------------
    // OperatorInterface
    // Local operator panel providing system status readout and manual override.
    // Typically a touchscreen HMI or physical panel with indicator lights.
    // -------------------------------------------------------------------------
    // ENGINEERING INPUTS:
    // ← PumpController status (this model)
    // ← AlarmSystem notification (this model)
    // ← HMI design tool: screen layout, button mapping (Wonderware / FactoryTalk)
    // ← Human factors / ergonomics standard (ISO 11064 control room design)
    // ENGINEERING OUTPUTS:
    // → PumpController override command (this model)
    // → Voyage Data Recorder (VDR): operator action log
    // → Vessel management system: maintenance event annotation
    part def OperatorInterface {
        // Input: status from controller
        port statusIn : ~StatusPort;
        // Input: notification from alarm system
        port notifyIn : ~AlarmPort;
        // Output: manual override command to controller
        port overrideOut : OverridePort;

        // True when the operator has asserted a manual override
        attribute overrideActive : Boolean;
    }
} // package BilgePump_Library
// CI proof-of-concept trigger — Fri May  8 08:15:31 CEST 2026


Package BilgePump_Library (86cb41d3-534e-43ac-9093-f03e3d3857be)


In [2]:
// =============================================================================
// BilgePump::Architecture
// Layer: Architecture
// Purpose: Instantiate Library part defs as part usages inside the top-level
// BilgePumpSystem composite part. All signal flows from the SysML
// block diagram are represented as connect statements here.
//
// Import chain: Architecture imports Library (one-way; no circular deps).
//
// Naming convention:
// - Part usages are lower-camel (sensor, controller, pumpA …)
// - Port references follow: <usage>.<portName>
// - Connections are directional: source port → destination port
// (where destination uses the conjugate ~Port def)
// =============================================================================

package BilgePump_Architecture {
    private import ScalarValues::*;
    public import BilgePump_Library::*;

    // =========================================================================
    // BilgePumpSystem
    // Top-level composite system.  Contains all 8 subsystem part usages and
    // the 11 connect statements that fully wire the signal/flow architecture
    // shown in the SysML IBD (internal block diagram).
    //
    // This part def is the subject bound in Requirements and Analysis layers.
    // It is intentionally definition-only here; the Analysis layer instantiates
    // it as a usage named 'sys' for constraint evaluation.
    // =========================================================================
    part def BilgePumpSystem {
        // ---------------------------------------------------------------------
        // System-level operational parameters — fault-tolerance analysis (FT-*)
        // These are not component attributes; they describe the operational
        // envelope of the entire system in the damage stability scenario.
        //
        // SOURCE (inflowRate_m3s): Naval architecture stability analysis tool
        // (Maxsurf / NAPA) — worst-case water ingress rate for the design flood
        // scenario.  Nominal design storm: 0.020 m³/s.
        // Catastrophic breach scenario (SOLAS damage case): up to 0.233 m³/s.
        //
        // SOURCE (criticalLevel_m): SOLAS Chapter II-1 structural margin.
        // Water level above which freeboard is compromised.
        // Conservative bound: 0.5 m above bilge floor.
        // ---------------------------------------------------------------------
        attribute inflowRate_m3s : Real = 0.020; // Design water ingress rate (m³/s)
        attribute criticalLevel_m : Real = 0.5; // Max tolerable level before SOLAS breach (m)

        // ---------------------------------------------------------------------
        // Part Usages
        // Each entry instantiates a Library part def as a named role within
        // the system boundary.
        // ---------------------------------------------------------------------

        // Water level sensing — bottom of the signal-flow hierarchy
        part sensor : BilgeWaterSensor {
            attribute waterLevel = 0.15; // nominal bilge sounding (m)
        }

        // Central control logic — receives level, drives pumps and alarm
        part controller : PumpController {
            attribute triggerLevel_m = 0.25; // activation level (m)
            attribute responseTime_s = 1.0; // breach→start (s)
        }

        // Dual-source power (main + emergency bus)
        part power : PowerSupply {
            attribute nominalVoltage = 440.0; // bus voltage (V)
            attribute redundancyActive = false; // emergency bus idle
        }

        // Primary bilge pump
        part pumpA : BilgePumpA {
            attribute flowRate = 0.025; // rated flow (m³/s)
            attribute efficiency = 0.82; // hydraulic efficiency
            attribute runHours = 120.0; // cumulative run hours
        }

        // Redundant bilge pump — independent power feed per DNV Pt.4 Ch.6
        part pumpB : BilgePumpB {
            attribute flowRate = 0.025; // rated flow (m³/s)
            attribute efficiency = 0.82; // hydraulic efficiency
            attribute runHours = 85.0; // cumulative run hours
            attribute isRedundant = true; // designated redundant unit
        }

        // Overboard discharge piping, strainer, and check valves
        part discharge : DischargeLine {
            attribute pipeLossFactor = 0.05; // Darcy-Weisbach λ
            attribute marpolLimit_m = 0.30; // MARPOL 73/78 max bilge-well level
        }

        // Audible / visual alarm unit
        part alarm : AlarmSystem {
            attribute activationDelay_s = 0.5; // IEC 60945 ≤ 2.0 s
            attribute isActive = false; // not sounding (nominal)
        }

        // Operator HMI panel — status display and manual override
        part ui : OperatorInterface {
            attribute overrideActive = false; // no override (nominal)
        }

        // ---------------------------------------------------------------------
        // Connect Statements — 11 signal / flow paths
        // Ordered to follow the data flow top-to-bottom as drawn in the SysML
        // block diagram:
        // Sensor → Controller → Pumps → Discharge
        // → Alarm → UI
        // ↔ UI (status / override)
        // Power  → Pumps
        // ---------------------------------------------------------------------

        // [1] Water level measurement: Sensor → Controller
        // This is the primary trigger signal for the entire system.
        // In vessel integration: also tapped by SCADA historian.
        connect sensor.levelOut to controller.levelIn;

        // [2] Pump A start/stop command: Controller → PumpA
        // Controller evaluates triggerLevel_m and responseTime_s before
        // issuing this command. Firmware generated from Simulink model.
        connect controller.pumpAControl to pumpA.controlIn;

        // [3] Pump B start/stop command: Controller → PumpB
        // Issued when Pump A is faulted or on manual override.
        connect controller.pumpBControl to pumpB.controlIn;

        // [4] Electrical power rail: PowerSupply → PumpA
        // Carries nominal voltage; isEmergency flag raised during bus transfer.
        connect power.powerOutA to pumpA.powerIn;

        // [5] Electrical power rail: PowerSupply → PumpB
        // Independent feed path required for redundancy per SOLAS II-1.
        connect power.powerOutB to pumpB.powerIn;

        // [6] Fluid discharge flow: PumpA → DischargeLine
        // Flow rate attribute is evaluated against inflow in Analysis layer.
        connect pumpA.flowOut to discharge.flowInA;

        // [7] Fluid discharge flow: PumpB → DischargeLine
        // Combined with PumpA flow for DischargeCapacityRequirement.
        connect pumpB.flowOut to discharge.flowInB;

        // [8] System status: Controller → OperatorInterface
        // Status string updated on every pump state change.
        // Also routed to VDR in vessel integration.
        connect controller.statusOut to ui.statusIn;

        // [9] Manual override: OperatorInterface → Controller
        // Bridge crew can force-start/stop pumps; controller logs override event.
        connect ui.overrideOut to controller.overrideIn;

        // [10] Alarm trigger: Controller → AlarmSystem
        // Fires when waterLevel > triggerLevel_m or fault detected.
        connect controller.alarmOut to alarm.alarmIn;

        // [11] Alarm notification: AlarmSystem → OperatorInterface
        // AlarmSystem annunciates locally; also mirrors to bridge BNWAS.
        connect alarm.notifyOut to ui.notifyIn;

        // ---------------------------------------------------------------------
        // State machine behavioral bindings
        // Links part instances to state machine definitions in StateMachine.sysml.
        // PumpControllerBehavior models 7 discrete controller operating states.
        // BilgePumpSystemBehavior models 5 system-level operational modes.
        // SOURCE: BilgePump_StateMachine (StateMachine.sysml)
        //
        // NOTE: These exhibits are commented out so that Architecture.sysml is
        // self-contained and kernel-publishable on its own. They reference types
        // defined in StateMachine.sysml; re-enable them only when StateMachine is
        // co-loaded/published in the same kernel session.
        // ---------------------------------------------------------------------
        // exhibit state controllerBehavior : PumpControllerBehavior;
        // exhibit state systemBehavior     : BilgePumpSystemBehavior;
    }
} // package BilgePump_Architecture


Package BilgePump_Architecture (5643d4bd-740d-40be-ab4e-cb6eef9d5afb)


In [3]:
// =============================================================================
// BilgePump::Requirements
// Layer: Requirements
// Purpose: Formal requirement definitions that the bilge pump system must
// satisfy. Each requirement def declares a subject (the system under
// verification) and an assert constraint body containing the logic
// that must evaluate to true.
//
// Import chain: Requirements imports Library and Architecture.
//
// Regulatory sources are cited in comments; these map to the external document
// inputs a real systems engineer would trace in a requirements management tool
// (e.g., IBM DOORS, Jama Connect, or Polarion).
//
// Traceability:
// All four requirements are asserted in Analysis::BilgePumpVerification.
// In a full MBSE toolchain (Cameo/Papyrus), each requirement def here would
// be linked to a verification method element and a test case element,
// forming the complete V-model traceability chain:
// Stakeholder Need → System Requirement → Design → Verification
// =============================================================================

package BilgePump_Requirements {
    private import ScalarValues::*;
    public import BilgePump_Library::*;
    public import BilgePump_Architecture::*;

    // =========================================================================
    // Claim-nature flags (Epic-1 extensibility prototype — step 1)
    //
    // Two metadata def stereotypes that let a requirement DECLARE its own nature,
    // so the machinery for that nature activates only when flagged (opt-in):
    //
    //   #Probabilistic  — the claim is genuinely probabilistic and MUST be
    //                     discharged by a sampling method (Monte Carlo / LHS).
    //                     The deterministic UQ sweep is NOT accepted. Applied to
    //                     NONE today: no in-scope claim is probabilistic, so the
    //                     extension correctly stays dormant (defaults OFF).
    //   #HazardBearing  — the claim carries a hazard→mitigation obligation.
    //                     Assigned OBJECTIVELY from the DNV corpus necessity
    //                     fields (necessity.prevents_major_accident /
    //                     prevents_fatality == true), never by convenience.
    //                     See docs/epic-1/requirements_mapping.py + the source
    //                     JSON necessity block.
    //
    // metadata def publishes on the pinned kernel today. Folded here (not a new
    // layer) to avoid a manifest/umbrella edit and to keep the flag defined in
    // the same package where it is applied.
    // =========================================================================
    metadata def Probabilistic {
        attribute epsilon : Real;   // acceptable exceedance probability
        attribute method  : String; // "MonteCarlo" | "LHS" — sweep is NOT accepted
    }
    metadata def HazardBearing {
        attribute hazardRef   : String; // Safety.sysml hazard id (H-xxx)
        attribute mitigatedBy : String; // FMEA verdict key (FM-xxx) that discharges it
    }

    // =========================================================================
    // RAAML provenance stereotypes (relocated from the former RAAML.sysml layer)
    //
    // These OMG RAAML v1.0 metadata defs were their own layer (RAAML.sysml,
    // RAAML-first import order). RAAML is render-only — 0 of the model's verdict
    // rows depend on it — so keeping it as a separate .sysml only raised the file
    // count and the umbrella-assembly burden. The STPA hazard→loss→UCA provenance
    // (mandatory case content) is preserved verbatim here; only the location moved.
    //
    // They live in Requirements (not Library) because Safety.sysml and FMEA.sysml
    // — the only consumers — already `public import BilgePump_Requirements::*`, so
    // dropping their `import BilgePump_RAAML::*` line is the whole migration. No
    // new import edges, and Requirements is loaded before Safety/FMEA, so the
    // stereotypes are defined before they are applied (kernel-valid ordering).
    //
    // Reference: OMG RAAML v1.0 https://www.omg.org/spec/RAAML/1.0
    // =========================================================================

    // Hazard — STAMP condition that, with a worst-case environment, leads to a
    // Loss. SOURCE: STPA Handbook (MIT Press, 2018) §3.2; OMG RAAML §7.3
    metadata def Hazard {
        attribute hazardId : String;       // e.g. "H-1"
        attribute description : String;     // Short hazard statement
        attribute lossRefs : String;        // Comma-separated Loss IDs, e.g. "L-0,L-1"
        attribute severity : String;        // catastrophic | critical | marginal | negligible
        attribute regulatoryRef : String;   // e.g. "SOLAS II-1 Reg.35"
        attribute sourceDoc : String;
        attribute section : String;
    }

    // Loss — stakeholder-unacceptable outcome. SOURCE: STPA Handbook §3.1; RAAML §7.2
    metadata def Loss {
        attribute lossId : String;          // e.g. "L-0"
        attribute description : String;
        attribute category : String;        // safety | environmental | mission | financial
        attribute severity : String;
        attribute regulatoryRef : String;
        attribute sourceDoc : String;
        attribute section : String;
    }

    // UCA — Unsafe Control Action. SOURCE: STPA Handbook §4.2; RAAML §7.5
    metadata def UCA {
        attribute ucaId : String;           // e.g. "UCA-001"
        attribute controlAction : String;   // e.g. "ActivatePumpA"
        attribute guideword : String;       // Not Provided | Wrong Value | Too Late | Too Long
        attribute context : String;
        attribute hazardRefs : String;
        attribute severity : String;
        attribute failureModeLink : String; // Link to FMEA failure mode id (optional)
        attribute transitionRef : String;   // StateMachine transition that eliminates it
        attribute sourceDoc : String;
        attribute section : String;
    }

    // FailureMode — FMEA record. SOURCE: MIL-STD-1629A; AIAG FMEA 4th Ed; RAAML §8.2
    metadata def FailureMode {
        attribute fmId : String;            // e.g. "FM-S-001"
        attribute component : String;
        attribute instance : String;
        attribute failureModeText : String;
        attribute failureEffect : String;
        attribute severity : Integer;       // 1–10 AIAG scale
        attribute occurrenceRating : Integer;
        attribute detection : Integer;
        attribute rpn : Integer;            // S×O×D
        attribute ucaRef : String;
        attribute hazardRef : String;
        attribute stateRef : String;        // StateMachine state (optional)
        attribute sourceDoc : String;
        attribute section : String;
    }

    // FaultTree — FTA top event tag. SOURCE: IEC 61025; OMG RAAML §9.3
    metadata def FaultTree {
        attribute ftId : String;            // e.g. "FT-001"
        attribute topEvent : String;
        attribute gate : String;            // OR | AND
        attribute cutSets : String;
        attribute sourceDoc : String;
        attribute section : String;
    }

    // SafetyRequirement — derived from a UCA or FMEA action (traceability bridge).
    // SOURCE: STPA Handbook §5.1 (Safety Constraints → Safety Requirements)
    metadata def SafetyRequirement {
        attribute srId : String;            // e.g. "SR-001"
        attribute derivedFrom : String;
        attribute rationale : String;
        attribute verificationMethod : String; // analysis | test | inspection | demonstration
        attribute sourceDoc : String;
        attribute section : String;
    }

    // =========================================================================
    // Requirement 1: Water Level Threshold
    //
    // Regulatory basis:
    // IMO MARPOL 73/78 Annex I, Regulation 22 — bilge water must be pumped
    // overboard before accumulation reaches a level that threatens stability.
    // Classification societies (DNV, Lloyd's) typically mandate automatic
    // pump activation at ≤ 300 mm (0.3 m) above the bilge floor.
    //
    // Verification method: Analysis case evaluates sensor.waterLevel at
    // steady-state nominal and at injected fault value.
    //
    // DOORS / Jama traceability ID: BPS-REQ-001
    // =========================================================================
    requirement def WaterLevelRequirement {
        subject sys : BilgePumpSystem;

        // The sensor's reported water level must never exceed 0.3 m when the
        // system is operating correctly (i.e., after controller has responded).
        doc
        /* The bilge water level shall not exceed 300 mm above the bilge
         * floor during normal automated operation.
         */

        require constraint {
            sys.sensor.waterLevel <= 0.3
        }
    }

    // =========================================================================
    // Requirement 2: Pump Redundancy
    //
    // Regulatory basis:
    // DNV Rules for Ships — Part 4 Chapter 6, Section 3:
    // "The bilge system shall be arranged with at least two pumps, each
    // capable of handling the required flow, with independent power supplies."
    // SOLAS Chapter II-1, Regulation 35: similar mandate.
    //
    // Verification method: Confirm pumpB.isRedundant flag is true and that
    // power.powerOutB is on an independent bus (modelled via isEmergency flag
    // in Analysis negative test).
    //
    // DOORS / Jama traceability ID: BPS-REQ-002
    // =========================================================================
    requirement def PumpRedundancyRequirement {
        // #HazardBearing — objectively derived: DNV corpus req #1 (>=2 pumping
        // units) carries necessity.prevents_major_accident == true AND
        // prevents_fatality == true (source JSON lines 28-30). Loss of the sole
        // dewatering unit → uncontrolled flooding → loss of vessel/life.
        #HazardBearing {
            hazardRef   = "H-1";      // uncontrolled flooding / loss of vessel
            mitigatedBy = "FM-C-003"; // failover-not-triggered FMEA verdict
        }
        subject sys : BilgePumpSystem;

        doc
        /* The bilge pump system shall include a designated redundant pump
         * (Pump B) with an independent power feed, capable of maintaining
         * full discharge capacity in the event of Pump A failure.
         */

        require constraint {
            sys.pumpB.isRedundant == true
        }
    }

    // =========================================================================
    // Requirement 3: Alarm Response Time
    //
    // Regulatory basis:
    // IEC 60945:2002 (Maritime navigation and radio communication equipment)
    // Section 4.3 — Alarm systems: audible/visual alarm must activate within
    // 2 seconds of detecting an alarm condition.
    // Also referenced in IMO MSC/Circ.982 (Guidelines for alarm management).
    //
    // Verification method: alarm.activationDelay_s is set to nominal value
    // (0.5 s) in the positive test and mutated to 3.0 s in the negative test.
    //
    // DOORS / Jama traceability ID: BPS-REQ-003
    // =========================================================================
    requirement def AlarmResponseRequirement {
        subject sys : BilgePumpSystem;

        doc
        /* The alarm system shall activate audible and visual annunciation
         * within 2.0 seconds of the controller issuing an alarm trigger,
         * in compliance with IEC 60945 Section 4.3.
         */

        require constraint {
            sys.alarm.activationDelay_s <= 2.0
        }
    }

    // =========================================================================
    // Requirement 4: Discharge Capacity
    //
    // Regulatory basis:
    // SOLAS Chapter II-1, Regulation 35: the bilge pumping system must be
    // capable of evacuating bilge water at a rate that prevents flooding.
    // The combined pump flow rate must exceed the maximum design inflow rate
    // (typically calculated from the worst-case damage stability scenario).
    //
    // Modelling note:
    // 'designInflow' is the maximum water ingress rate from the damage
    // stability calculation (naval architecture tool output — e.g., Maxsurf
    // Stability or NAPA). In this model it is declared as an attribute on the
    // system subject so the Analysis layer can bind it to the constraint.
    //
    // Verification method: PumpFlowPhysics constraint in Analysis layer
    // evaluates (pumpA.flowRate + pumpB.flowRate) * efficiency * (1 - pipeLoss)
    // against a bound designInflow value.
    //
    // DOORS / Jama traceability ID: BPS-REQ-004
    // =========================================================================
    requirement def DischargeCapacityRequirement {
        // #HazardBearing — objectively derived: DNV corpus req #4 (unit
        // composition + combined-capacity duty) carries
        // necessity.prevents_major_accident == true AND prevents_fatality == true
        // (source JSON lines 710-712). Insufficient discharge vs design inflow →
        // flooding faster than dewatering → loss of vessel/life.
        #HazardBearing {
            hazardRef   = "H-1";      // uncontrolled flooding / loss of vessel
            mitigatedBy = "FM-PB-001"; // redundant-pump failure FMEA verdict
        }
        subject sys : BilgePumpSystem;

        // Design inflow rate from damage stability analysis (m³/s).
        // SOURCE: naval architecture tool (Maxsurf / NAPA stability module)
        attribute designInflow : Real;

        doc
        /* The combined discharge flow rate of Pump A and Pump B, after
         * accounting for hydraulic efficiency and pipe losses, shall meet
         * or exceed the design inflow rate derived from the damage stability
         * analysis, as required by SOLAS II-1 Regulation 35.
         */

        require constraint {
            sys.pumpA.flowRate + sys.pumpB.flowRate >= designInflow
        }
    }
    //
    // =========================================================================
    // Requirement 5: Controller Activation Timing
    //
    // Regulatory basis:
    // Derived from STPA UCA-001 and UCA-005 safety constraints (SR-001/SR-005
    // in Safety.sysml). Formalised here at the system-requirements layer
    // with a directly measurable bound.
    // Simulink discrete-event timing analysis (SIM-CTRL-001 §3.1) validates:
    // Nominal:           responseTime_s = 1.0 s → PASS
    // High CPU load:     responseTime_s = 2.8 s → PASS
    // Watchdog recovery: responseTime_s = 4.9 s → PASS (marginal)
    // Software hang:     responseTime_s = 9999.0 s → FAIL (FM-C-001)
    //
    // Verification method: BilgePumpTimingVerification analysis def.
    // Nominal: responseTime_s = 1.0 → SATISFIED.
    // Negative: responseTime_s = 9999.0 → VIOLATED (see FMEA_ControllerHang).
    //
    // State machine link: MONITORING → PUMP_A_ACTIVE transition guard.
    // See PumpControllerBehavior in StateMachine.sysml.
    //
    // DOORS / Jama traceability ID: BPS-REQ-005
    // =========================================================================
    requirement def ControllerActivationTimingRequirement {
        subject sys : BilgePumpSystem;
        //
        doc
        /* The PumpController shall issue a pump activation command within
         * 5.0 seconds of detecting a water level at or above triggerLevel_m.
         * Derived from STPA SR-001/SR-005; validated by Simulink timing
         * analysis SIM-CTRL-001 §3.1.
         */
        //
        require constraint {
            sys.controller.responseTime_s <= 5.0
        }
    }
    //
    // =========================================================================
    // Requirement 6: Failover Switch Timing
    //
    // Regulatory basis:
    // Derived from STPA UCA-004 safety constraint (SR-004 in Safety.sysml).
    // This requirement closes a gap in the original 4-requirement set:
    // BPS-REQ-002 (PumpRedundancyRequirement) verifies structural redundancy
    // (pumpB.isRedundant == true) but does NOT verify the TIMING of failover.
    // SIM-CTRL-001 §3.2 provides the timing bound:
    // Normal failover:   failoverTime_s = 0.8 s → PASS
    // Delayed detection: failoverTime_s = 2.7 s → PASS
    // No failover:       failoverTime_s = 9999.0 s → FAIL (FM-C-003)
    //
    // Verification method: BilgePumpTimingVerification analysis def.
    // Nominal: failoverTime_s = 0.8 s → SATISFIED.
    // Negative: failoverTime_s = 9999.0 s → VIOLATED (FMEA_FailoverNotTriggered).
    //
    // State machine link: PUMP_A_ACTIVE → FAILOVER transition guard.
    // See PumpControllerBehavior in StateMachine.sysml.
    //
    // DOORS / Jama traceability ID: BPS-REQ-006
    // =========================================================================
    requirement def FailoverSwitchTimingRequirement {
        subject sys : BilgePumpSystem;
        //
        doc
        /* The PumpController shall issue a Pump B activation command within
         * 3.0 seconds of detecting a Pump A fault condition.
         * Derived from STPA SR-004 and Simulink failover timing analysis
         * SIM-CTRL-001 §3.2. Closes the gap in BPS-REQ-002 which verifies
         * structural redundancy but not failover timing.
         */
        //
        require constraint {
            sys.controller.failoverTime_s <= 3.0
        }
    }
    //
    // =========================================================================
    // Fault-Tolerance Requirements (FT-*)
    //
    // These requirements decompose the abstract safety property:
    // "The bilge pump system shall remain safe under all conditions within
    // its defined operating envelope."
    //
    // Motivation: the original 6 requirements (REQ-001..006) each check a
    // single attribute at a single point in time.  They CANNOT detect:
    // - sensor accuracy consuming the margin between trigger and MARPOL limit
    // - the efficiency + pipe-loss product undermining REQ-004's simple sum
    // - the timing chain (response + alarm) vs the physical overflow window
    // - adversarial parameter combinations where all 6 pass yet system fails
    //
    // All FT-* requirements are also checked by the Z3 formal analyser
    // (bilgepump/formal_analysis.py), which discovers the parametric envelopes
    // and counterexamples the SysML point-in-time engine cannot find.
    // =========================================================================

    // =========================================================================
    // FT-001: Sensor Accuracy Bound
    //
    // Gap closed: BPS-REQ-001 checks sys.sensor.waterLevel ≤ 0.3.
    // But waterLevel is the REPORTED value; the true level can be up to
    // (waterLevel + accuracy_m) if the sensor under-reads.  This requirement
    // closes the gap by adding the accuracy margin into the bound.
    //
    // Regulatory basis: sensor calibration certificate + IMO MARPOL 73/78 —
    // the 300 mm limit applies to the ACTUAL level, not the instrument reading.
    //
    // DOORS / Jama traceability ID: BPS-FT-001
    // =========================================================================
    requirement def SensorAccuracyBoundRequirement {
        subject sys : BilgePumpSystem;

        doc
        /* The reported water level plus the sensor accuracy margin shall
         * not exceed 300 mm.  This ensures the actual bilge level remains
         * within the MARPOL limit even under worst-case sensor under-reading.
         */

        require constraint {
            sys.sensor.waterLevel + sys.sensor.accuracy_m <= 0.3
        }
    }

    // =========================================================================
    // FT-002: Effective Discharge Capacity
    //
    // Gap closed: BPS-REQ-004 checks (flowA + flowB) ≥ designInflow, but the
    // PumpFlowPhysics constraint shows the net flow is actually:
    // Q_net = (flowA × η_A + flowB × η_B) × (1 − λ)
    // A pump pair that satisfies REQ-004 may still fail to overcome inflow
    // if efficiency is low and pipe losses are high.
    //
    // Regulatory basis: SOLAS II-1 Reg. 35 — required flow must be achievable
    // after accounting for all hydraulic losses.
    //
    // Modelling note: η_A and η_B are per-pump efficiencies, which may differ
    // (e.g., after wear).  designInflow is the same damage-stability value
    // used by REQ-004.
    //
    // DOORS / Jama traceability ID: BPS-FT-002
    // =========================================================================
    requirement def EffectiveDischargeCapacityRequirement {
        subject sys : BilgePumpSystem;

        // Design inflow rate from damage stability analysis (m³/s).
        // Same source as REQ-004: naval architecture tool (Maxsurf / NAPA).
        attribute designInflow : Real;

        doc
        /* The net discharge flow, after accounting for per-pump hydraulic
         * efficiency and pipe friction losses, shall meet or exceed the
         * design inflow rate.  This is more conservative than BPS-REQ-004,
         * which only checks the raw sum of pump flow rates.
         */

        require constraint {
            (sys.pumpA.flowRate * sys.pumpA.efficiency +
                sys.pumpB.flowRate * sys.pumpB.efficiency) *
            (1.0 - sys.discharge.pipeLossFactor) >=
            designInflow
        }
    }

    // =========================================================================
    // FT-003: Trigger Level Accuracy
    //
    // Gap closed: BPS-REQ-001 and BPS-REQ-005 both ignore sensor accuracy.
    // If the controller uses triggerLevel_m to decide when to start the pump,
    // and the sensor can under-read by accuracy_m, then the pump may not start
    // until the true level is (triggerLevel_m + accuracy_m) — which must still
    // be below the 300 mm MARPOL limit.
    //
    // Concrete scenario: triggerLevel_m=0.25, accuracy_m=0.05 → trigger
    // fires at apparent 0.25, but true level could be 0.30 (at the limit).
    //
    // DOORS / Jama traceability ID: BPS-FT-003
    // =========================================================================
    requirement def TriggerLevelAccuracyRequirement {
        subject sys : BilgePumpSystem;

        doc
        /* The controller trigger threshold plus the sensor accuracy margin
         * shall not exceed 300 mm.  This ensures pump activation occurs
         * before the actual water level reaches the MARPOL limit, even
         * under worst-case sensor under-reading.
         */

        require constraint {
            sys.controller.triggerLevel_m + sys.sensor.accuracy_m <= 0.3
        }
    }

    // =========================================================================
    // FT-004: End-to-End Response Time vs Overflow Window
    //
    // Gap closed: BPS-REQ-005 checks responseTime_s ≤ 5.0 s and BPS-REQ-003
    // checks alarmDelay_s ≤ 2.0 s.  Neither requirement verifies whether the
    // COMBINED timing chain (controller response + alarm activation) is short
    // enough relative to the physical overflow window — defined as the time
    // from water level at sensor threshold to SOLAS critical level, at the
    // current inflow rate.
    //
    // Formulation (division-free, equivalent form):
    // (responseTime_s + activationDelay_s) × inflowRate_m3s
    // ≤ criticalLevel_m − sensor.waterLevel
    //
    // Interpretation: the volume added to the bilge during the total response
    // delay must not exceed the remaining safe headroom.
    //
    // NOTE: inflowRate_m3s and criticalLevel_m are system-level attributes
    // added to BilgePumpSystem (Architecture.sysml) for this requirement.
    //
    // DOORS / Jama traceability ID: BPS-FT-004
    // =========================================================================
    requirement def EndToEndResponseRequirement {
        subject sys : BilgePumpSystem;

        doc
        /* The product of the total system response time (controller
         * activation delay plus alarm activation delay) and the water
         * inflow rate shall not exceed the remaining safe headroom
         * (critical level minus current water level).  This ensures
         * the timing chain is feasible for the physical flooding scenario.
         */

        require constraint {
            (sys.controller.responseTime_s + sys.alarm.activationDelay_s) * sys.inflowRate_m3s <=
            sys.criticalLevel_m - sys.sensor.waterLevel
        }
    }

    // =========================================================================
    // OOR-001: Override Ordering Requirement
    //
    // Gap closed: StateMachine.sysml has no guard preventing operator override
    // before the alarm notification arrives at the UI (alarm.notifyOut →
    // ui.notifyIn, connection [11]).  Z3 Level 6 (formal_analysis.py) found a
    // satisfiable trace where t_operator_override < t_alarm_notify_ui, meaning
    // a crew member could act on an override before seeing the alarm.
    //
    // This requirement closes that gap with a static snapshot constraint:
    // "if override is active, the alarm must already be active"
    // This is a conservative instantaneous approximation.  Full temporal proof
    // (for all traces, not just a single snapshot) requires Z3 Level 6 or a
    // model checker such as nuXmv / PRISM fed the StateMachine.sysml states.
    //
    // Regulatory basis:
    // IMO MSC/Circ.982 (Alarm Management Guidelines):
    // "Operator action must only be available after alarm acknowledgement."
    // DNV Rules Pt.4 Ch.9 (Alarm Systems): alarm acknowledgement prerequisite
    // before operator can suppress or override alarm-triggered actuation.
    //
    // State machine fix required: add a guard to controller.overrideIn port
    // that rejects commands unless sys.alarm.isActive == true.
    //
    // DOORS / Jama traceability ID: BPS-OOR-001
    // =========================================================================
    requirement def OverrideOrderingRequirement {
        subject sys : BilgePumpSystem;

        doc
        /* If the operator override is active, the alarm system must already
         * be active.  This ensures crew acts only after receiving the alarm
         * notification, preventing premature override before hazard awareness.
         * Derived from STPA Level-6 temporal ordering gap (Z3 formal_analysis).
         */

        require constraint {
            // "override active" implies "alarm active"
            // Instantaneous approximation of the temporal ordering property:
            // t_operator_override ≥ t_alarm_notify_ui
            sys.ui.overrideActive == false or sys.alarm.isActive == true
        }
    }
} // package BilgePump_Requirements


Package BilgePump_Requirements (51f7fbe3-0d9d-4490-ad19-19adef4fbbed)


In [4]:
// =============================================================================
// BilgePump::Analysis
// Layer: Analysis
// Purpose: Physics constraint definitions and the verification cases (the
// "test runner") that evaluate every system requirement.
//
// Import chain: Analysis imports Library, Architecture, and Requirements.
// Library <- Architecture <- Requirements <- Analysis   (no cycles)
//
// HOW THE VERIFICATION CASES WORK (kernel-evaluable form)
// -------------------------------------------------------
// The SysML v2 Pilot kernel does not execute `assert requirement` statements,
// and it cannot override an attribute value that Architecture already binds
// (e.g. `attribute waterLevel = 0.15;`).  To obtain machine-checkable PASS/FAIL
// verdicts that the pipeline can capture with `%eval`, each requirement is
// re-expressed here as a computed Boolean attribute on a concrete subject:
//
// part bpVerification : BilgePumpSystem {
// attribute BPS_REQ_001_waterLevel : Boolean = sensor.waterLevel <= 0.3;
// }
//
// The subject `bpVerification` is a usage of BilgePumpSystem, so it INHERITS
// every nominal attribute value declared in Architecture.sysml.  Attributes
// that Architecture leaves unbound (controller.failoverTime_s, sensor.accuracy_m)
// are supplied here via nested redefinition `attribute :>> name = value;`.
//
// Each Boolean is queryable headlessly:
// %eval BilgePump_Analysis::bpVerification.BPS_REQ_001_waterLevel
// -> LiteralBoolean true     (SATISFIED)
// The publish pipeline emits one %eval per Boolean and records the verdict in
// the `sysml_assertions` database table (see materialize_sysml_values.py).
//
// Negative (fault) verdicts live in FMEA.sysml; the deterministic worst-case
// margin probe (formerly the UQ.sysml layer) is folded in at the end of this
// package. The two physics constraint defs below are reused by both.
// =============================================================================

package BilgePump_Analysis {
    private import ScalarValues::*;
    public import BilgePump_Library::*;
    public import BilgePump_Architecture::*;
    public import BilgePump_Requirements::*;

    // =========================================================================
    // StateTransitionTimingPhysics  -- state machine transition timing budget
    //
    // Models the timing chain from sensor threshold crossing to pump activation
    // command (Simulink discrete-event model SIM-CTRL-001 §3.1):
    // totalTime_s = samplingLatency_s + processingTime_s + commandLatency_s
    // and asserts totalTime_s <= maxResponseTime_s.
    //
    // Linked requirement: BPS-REQ-005 (ControllerActivationTimingRequirement)
    // Linked state:       MONITORING -> PUMP_A_ACTIVE transition (StateMachine.sysml)
    // =========================================================================
    constraint def StateTransitionTimingPhysics {
        attribute samplingLatency_s : Real; // Sensor poll interval (s)
        attribute processingTime_s : Real; // Firmware processing time (s)
        attribute commandLatency_s : Real; // Output command transmission latency (s)
        attribute maxResponseTime_s : Real; // Maximum allowed response time (s) [BPS-REQ-005]
        attribute totalTime_s : Real; // Computed total response time

        assert constraint {
            totalTime_s == samplingLatency_s + processingTime_s + commandLatency_s and
            totalTime_s <= maxResponseTime_s
        }
    }

    // =========================================================================
    // PumpFlowPhysics  -- net discharge capacity physics constraint
    //
    // Q_net = (Q_A + Q_B) * eta * (1 - lambda)   and   Q_net >= Q_inflow
    //
    // EXTERNAL TOOL BINDINGS (hypothetical):
    // <- CFD (ANSYS Fluent / OpenFOAM): hydraulic efficiency eta
    // <- P&ID hydraulic calc (Caesar II / PIPESIM): pipe loss factor lambda
    // <- Maxsurf / NAPA stability module: design inflow Q_inflow
    // =========================================================================
    constraint def PumpFlowPhysics {
        attribute flowRateA : Real; // Pump A flow rate (m^3/s)
        attribute flowRateB : Real; // Pump B flow rate (m^3/s)
        attribute efficiency : Real; // Hydraulic efficiency eta (0.0-1.0)
        attribute pipeLossFactor : Real; // Pipe friction factor lambda (0.0-1.0)
        attribute designInflow : Real; // Required minimum discharge (m^3/s)
        attribute netFlow : Real; // Computed net effective discharge

        assert constraint {
            netFlow == (flowRateA + flowRateB) * efficiency * (1.0 - pipeLossFactor) and
            netFlow >= designInflow
        }
    }

    // =========================================================================
    // bpVerification  -- functional verification case (POSITIVE / nominal)
    //
    // Subject inherits all nominal attribute values from Architecture.sysml.
    // Each requirement is encoded as a computed Boolean (SATISFIED == true).
    //
    // Nominal values (from Architecture): sensor.waterLevel=0.15,
    // pumpA/B.flowRate=0.025, efficiency=0.82, discharge.pipeLossFactor=0.05,
    // alarm.activationDelay_s=0.5, pumpB.isRedundant=true.
    // netFlow = (0.025+0.025) * 0.82 * (1-0.05) = 0.03895 m^3/s >= 0.030  OK
    // =========================================================================
    part bpVerification : BilgePumpSystem {
        // Design inflow from damage stability analysis (Maxsurf / NAPA export)
        attribute designInflow : Real = 0.030;
        // Net effective discharge (PumpFlowPhysics evaluated on nominal values)
        attribute netFlow : Real =
            (pumpA.flowRate + pumpB.flowRate) * pumpA.efficiency * (1.0 - discharge.pipeLossFactor);

        // TEST BPS-REQ-001: water level must not exceed 0.3 m  [MARPOL 73/78]
        attribute BPS_REQ_001_waterLevel : Boolean = sensor.waterLevel <= 0.3;
        // TEST BPS-REQ-002: Pump B must be the designated redundant unit  [DNV Pt.4 Ch.6]
        attribute BPS_REQ_002_redundancy : Boolean = pumpB.isRedundant == true;
        // TEST BPS-REQ-003: alarm activation within 2.0 s  [IEC 60945 §4.3]
        attribute BPS_REQ_003_alarm : Boolean = alarm.activationDelay_s <= 2.0;
        // TEST BPS-REQ-004: combined discharge meets design inflow  [SOLAS II-1 Reg.35]
        attribute BPS_REQ_004_discharge : Boolean = pumpA.flowRate + pumpB.flowRate >= designInflow;
        // Physics: net effective discharge meets design inflow (efficiency + losses)
        attribute physicsCheck : Boolean = netFlow >= designInflow;
    }

    // =========================================================================
    // bpTimingVerification  -- timing verification case (POSITIVE / nominal)
    //
    // Verifies the two timing requirements (BPS-REQ-005, BPS-REQ-006) and
    // re-checks alarm timing (BPS-REQ-003) in the same harness.
    // failoverTime_s is unbound in Architecture, so it is supplied here.
    //
    // Nominal: responseTime_s=1.0 (SIM-CTRL-001 §3.1), failoverTime_s=0.8
    // (SIM-CTRL-001 §3.2), alarm.activationDelay_s=0.5.
    // =========================================================================
    part bpTimingVerification : BilgePumpSystem {
        // Failover time not bound in Architecture -> supply the nominal value
        part :>> controller {
            attribute :>> failoverTime_s = 0.8; // SIM-CTRL-001 §3.2 normal failover
        }

        // Timing budget decomposition (StateTransitionTimingPhysics, nominal)
        // 0.1 s sampling + 0.9 s processing + 0.0 s command = 1.0 s <= 5.0 s
        attribute timingBudgetTotal_s : Real = 0.1 + 0.9 + 0.0;
        attribute timingBudgetOk : Boolean = timingBudgetTotal_s <= 5.0;

        // TEST BPS-REQ-005: controller activation within 5.0 s of threshold crossing
        attribute BPS_REQ_005_activation : Boolean = controller.responseTime_s <= 5.0;
        // TEST BPS-REQ-006: failover switch within 3.0 s of Pump A fault
        attribute BPS_REQ_006_failover : Boolean = controller.failoverTime_s <= 3.0;
        // TEST BPS-REQ-003 (re-asserted in timing context): alarm within 2.0 s
        attribute BPS_REQ_003_alarmTiming : Boolean = alarm.activationDelay_s <= 2.0;
    }

    // =========================================================================
    // bpFaultToleranceVerification  -- fault-tolerance verification (POSITIVE)
    //
    // Verifies the FT-* requirements (BPS-FT-001..004) and OOR-001 that
    // decompose the abstract property "the system shall remain safe under all
    // conditions within its operating envelope".
    //
    // sensor.accuracy_m (unbound in Architecture) supplied here = 0.03
    // (±3 cm sensor class, IEC 60770-1).  inflowRate_m3s=0.020 and
    // criticalLevel_m=0.5 are inherited from Architecture (BilgePumpSystem).
    // failoverTime_s supplied for completeness.
    //
    // FT-001: 0.15 + 0.03 = 0.18 <= 0.30  OK
    // FT-002: (0.025*0.82 + 0.025*0.82) * (1-0.05) = 0.03895 >= 0.030  OK
    // FT-003: 0.25 + 0.03 = 0.28 <= 0.30  OK (narrow margin)
    // FT-004: (1.0 + 0.5) * 0.020 = 0.030 <= 0.5 - 0.15 = 0.35  OK
    // OOR-001: overrideActive(false) == false  OR  alarm.isActive  -> true  OK
    // =========================================================================
    part bpFaultToleranceVerification : BilgePumpSystem {
        // Sensor accuracy not bound in Architecture -> supply the class value
        part :>> sensor {
            attribute :>> accuracy_m = 0.03; // ±3 cm (IEC 60770-1 accuracy class)
        }
        part :>> controller {
            attribute :>> failoverTime_s = 0.8;
        }

        attribute designInflow : Real = 0.030;
        attribute netFlow : Real =
            (pumpA.flowRate * pumpA.efficiency + pumpB.flowRate * pumpB.efficiency) *
            (1.0 - discharge.pipeLossFactor);

        // TEST FT-001: reported level + sensor accuracy stays within MARPOL limit
        attribute BPS_FT_001_sensorAccuracy : Boolean =
            sensor.waterLevel + sensor.accuracy_m <= 0.3;
        // TEST FT-002: net effective discharge (per-pump efficiency) >= design inflow
        attribute BPS_FT_002_effDischarge : Boolean = netFlow >= designInflow;
        // TEST FT-003: trigger level + sensor accuracy stays within MARPOL limit
        attribute BPS_FT_003_triggerAccuracy : Boolean =
            controller.triggerLevel_m + sensor.accuracy_m <= 0.3;
        // TEST FT-004: timing chain feasible vs physical overflow window
        attribute BPS_FT_004_responseWindow : Boolean =
            (controller.responseTime_s + alarm.activationDelay_s) * inflowRate_m3s <=
            criticalLevel_m - sensor.waterLevel;
        // TEST OOR-001: override only after alarm active (Z3 Level-6 gap closure)
        attribute BPS_OOR_001_overrideOrdering : Boolean =
            ui.overrideActive == false or alarm.isActive == true;
    }

    // =========================================================================
    // V-MODEL TRACEABILITY (documentation)
    //
    // SysML v2 satisfy/verify relationships complete the V-model, but the Pilot
    // kernel in use rejects the `satisfy <RequirementDef> by <Type>;` and
    // `verify <RequirementDef> by <AnalysisDef>;` forms (they require a
    // requirement USAGE and a feature, not definitions/types).  The traceability
    // is therefore recorded here as documentation; the executable evidence is
    // the computed-Boolean verdicts above (positive), in FMEA.sysml (negative),
    // and the folded UQ margin probe below (deterministic worst-case).
    //
    // satisfy WaterLevelRequirement                 by BilgeWaterSensor
    // satisfy PumpRedundancyRequirement             by BilgePumpB
    // satisfy AlarmResponseRequirement              by AlarmSystem
    // satisfy DischargeCapacityRequirement          by BilgePumpA, BilgePumpB, DischargeLine
    // satisfy EffectiveDischargeCapacityRequirement by BilgePumpA, BilgePumpB, DischargeLine
    // satisfy ControllerActivationTimingRequirement by PumpController
    // satisfy FailoverSwitchTimingRequirement       by PumpController
    // satisfy SensorAccuracyBoundRequirement        by BilgeWaterSensor
    // satisfy TriggerLevelAccuracyRequirement       by BilgeWaterSensor, PumpController
    // satisfy EndToEndResponseRequirement           by PumpController, AlarmSystem
    // satisfy OverrideOrderingRequirement           by PumpController, AlarmSystem, OperatorInterface
    //
    // verify  WaterLevelRequirement                 by bpVerification
    // verify  PumpRedundancyRequirement             by bpVerification
    // verify  AlarmResponseRequirement              by bpVerification, bpTimingVerification
    // verify  DischargeCapacityRequirement          by bpVerification
    // verify  EffectiveDischargeCapacityRequirement by bpFaultToleranceVerification
    // verify  ControllerActivationTimingRequirement by bpTimingVerification
    // verify  FailoverSwitchTimingRequirement       by bpTimingVerification
    // verify  SensorAccuracyBoundRequirement        by bpFaultToleranceVerification
    // verify  TriggerLevelAccuracyRequirement       by bpFaultToleranceVerification
    // verify  EndToEndResponseRequirement           by bpFaultToleranceVerification
    // verify  OverrideOrderingRequirement           by bpTimingVerification
    // =========================================================================

    // =========================================================================
    // CONTROL-ALLOCATION RELATIONS (Epic-1 extensibility prototype — step 2)
    //
    // Wire the native `allocate` construct (proven kernel-valid by the issue #9
    // spike: `allocate bpVerification.controller to bpVerification.pumpA;`).
    // Duty/function is allocated to the pump component(s) in the single
    // bpVerification : BilgePumpSystem config. Mapping evidence + shortlist:
    // docs/epic-1/requirements_mapping.py (CA bucket = 12 candidates).
    //
    // These are structural allocation edges, not %eval verdicts — they land in
    // the AllocationUsage metaclass table (queryable via the REST elements API
    // by @type), NOT in sysml_values / sysml_assertions.
    // =========================================================================

    // CA #1 (§8.1.1) redundancy duty: the dewatering control action is allocated
    // to BOTH pumping units (>= 2 units bear the function).
    allocate bpVerification.controller to bpVerification.pumpA;
    allocate bpVerification.controller to bpVerification.pumpB;

    // CA #3 (§8.1.1) independence: each unit is allocated its own power feed
    // (independent drive — powerOutA / powerOutB are separate paths).
    allocate bpVerification.power to bpVerification.pumpA;
    allocate bpVerification.power to bpVerification.pumpB;

    // CA #10 (§8.2.1) per-unit capacity duty (>= 2 m/s): primary unit allocated
    // to the discharge line (carries the combined-capacity duty).
    allocate bpVerification.pumpA to bpVerification.discharge;

    // CA #12 (§8.2.2) apportionment (smaller/redundant unit >= 1/3 combined):
    // the secondary unit is separately allocated its apportioned discharge duty.
    // (The numeric >= 1/3 ratio remains a parametric bound; this edge records the
    //  shared combined-capacity allocation of the second unit.)
    allocate bpVerification.pumpB to bpVerification.discharge;

    // =========================================================================
    // FOLDED UQ MARGIN PROBE (was the standalone UQ.sysml layer)
    //
    // The former UQ layer ran a 10-point DETERMINISTIC sigma sweep as its own
    // evaluated layer. Deterministic points are not genuine uncertainty
    // quantification, so per the Epic-1 evaluability verdict the sweep is retired
    // as a layer and ONE credible worst-case margin probe is folded here: the
    // combined −3σ flow AND −3σ efficiency point (the earliest-warning case that
    // still SATISFIES, ~1.7 % margin). It keeps the robustness signal without a
    // separate layer.
    //
    // DORMANCY RULE: real (sampling-based) UQ is woken from dormancy ONLY when a
    // system's particular rules necessitate a genuinely probabilistic discharge —
    // i.e. when a requirement is tagged #Probabilistic (Requirements.sysml). That
    // flag is the switch; it has 0 applications today (OFF by default). Actual
    // Monte-Carlo / LHS sampling is Epic-2, not this deterministic probe.
    // =========================================================================
    attribute uqDesignInflow : Real = 0.030;
    // Combined −3σ flow (0.019) and −3σ efficiency (0.73): netFlow ≈ 0.0305 ≥ 0.030
    attribute UQ_MARGIN_netFlow : Real = (0.019 + 0.025) * 0.73 * (1.0 - 0.05);
    attribute UQ_MARGIN_PROBE : Boolean = UQ_MARGIN_netFlow >= uqDesignInflow;

    // =========================================================================
    // bpSimEvidence  -- MOCK "FMU OUTPUT" RECEIVER (Epic-1 point 4, EXECUTABLE)
    //
    // Concept: the SysML package is assumed to RECEIVE simulation updates. A mock
    // "FMU output" (NOT an FMU) acts as the receiver of system data from a
    // simulation, registers it into the model, and the requirement checks below
    // report pass/fail on the received values — on the SAME %eval path as the
    // other verdict rows, so these are real sysml_assertions rows, not docs.
    //
    // Provenance: the values are the "latest simulation update" emitted by
    // docs/epic-1/bilge_dynamics_spike.py (the bilge-well ODE) into
    // examples/bilgepump/sim_evidence.json, transcribed here to 3 dp. That ODE
    // CONSUMES this model's own nominals (flowRate 0.025, efficiency 0.82,
    // pipeLoss 0.05, MARPOL 0.30), so these verdicts test THIS system — the
    // coupling guard that separates evidence from theater.
    //
    // Cascade: the received simMaxLevel_m flows into SIM_LEVEL_001, which ties to
    // the MARPOL discharge-capacity claim (BPS-REQ-004 / BPS-REQ-001). Change the
    // received number and the verdict changes — proven by SIM_LEVEL_001_NEG,
    // which feeds the SAME expression the UNDERSIZED-pump mock (1.167 m) and MUST
    // evaluate false. That is the falsifiability proof, executed in the DB: the
    // row is driven by the sim value, not hardcoded true.
    // =========================================================================
    part bpSimEvidence : BilgePumpSystem {
        // --- received simulation update: nominal design (sim_evidence.json) ---
        attribute simMaxLevel_m : Real = 0.286;          // nominal max well level
        attribute simResponseTime_s : Real = 1.1;        // controller response time
        attribute simFailoverMaxLevel_m : Real = 0.248;  // max level during failover phase
        // --- falsifiability mock: undersized pump (50 % capacity) -------------
        attribute simMaxLevel_undersized_m : Real = 1.167; // overflows MARPOL

        // SIM-LEVEL-001: dynamic MARPOL check driven by the received sim level
        //                (ties to BPS-REQ-004 / BPS-REQ-001 discharge-capacity claim)
        attribute SIM_LEVEL_001 : Boolean = simMaxLevel_m <= 0.30;
        // SIM-RESP-005: dynamic response-time check (ties to BPS-REQ-005)
        attribute SIM_RESP_005 : Boolean = simResponseTime_s <= 5.0;
        // SIM-FO-006: failover consequence check (ties to BPS-REQ-006 timing)
        attribute SIM_FO_006 : Boolean = simFailoverMaxLevel_m <= 0.30;
        // SIM-LEVEL-001-NEG: SAME expression, fed the undersized mock -> MUST be
        // false. Proves SIM_LEVEL_001 is computed from the input, not a constant.
        attribute SIM_LEVEL_001_NEG : Boolean = simMaxLevel_undersized_m <= 0.30;
    }
} // package BilgePump_Analysis


Package BilgePump_Analysis (ce24261f-2aae-4a19-90c8-2f2c9d59a097)


In [5]:
// =============================================================================
// BilgePump::Safety
// Layer: Safety Analysis (STPA)
// Purpose: System-Theoretic Process Analysis (STPA) artifacts for the bilge
// pump system. Defines Losses, Hazards, and Unsafe Control Actions
// (UCAs) as formal requirement def blocks annotated with OMG RAAML
// metadata stereotypes.
//
// This layer EXTENDS the existing 4-layer model — it does not replace anything.
//
// Import chain:
// BilgePump::RAAML           — metadata def stereotypes (Hazard, UCA, Loss)
// BilgePump::Library         — part def and attribute def types
// BilgePump::Architecture    — BilgePumpSystem, part usage instances
// BilgePump::Requirements    — existing 4 requirement defs (for cross-reference)
//
// SOURCE DOCUMENTS:
// STPA-BPS-001 — Hazard and Loss identification
// STPA-BPS-002 — Unsafe Control Actions table
// STPA-BPS-003 — Loss scenarios (consumed by AnalysisMapper → FMEA.sysml)
//
// RAAML ANNOTATION COMPATIBILITY:
// #Hazard, #Loss, #UCA, #SafetyRequirement annotations require SysML v2
// Pilot API JAR ≥ 2022-06. If the API rejects metadata annotations, remove
// the #Annotation lines only — the requirement def blocks compile standalone.
//
// VERIFICATION:
// Positive test (nominal): All UCA requirement defs SATISFIED under nominal
// attribute bindings in BilgePumpVerification (Analysis.sysml).
// Negative tests: See FMEA.sysml for failure-mode analysis cases that
// inject fault values and confirm UCA requirement defs are VIOLATED.
// =============================================================================

package BilgePump_Safety {
    private import ScalarValues::*;
    // RAAML stereotypes (Hazard/Loss/UCA/…) were folded into Requirements.sysml;
    // they arrive via the existing BilgePump_Requirements import below.
    public import BilgePump_Library::*;
    public import BilgePump_Architecture::*;
    public import BilgePump_Requirements::*;

    // =========================================================================
    // STPA LOSSES — top-level unacceptable outcomes
    // Annotated with #Loss (OMG RAAML Loss stereotype)
    // These are not verifiable constraints; they are reference nodes that
    // UCAs and Hazards trace back to via hazardRefs / lossRefs metadata.
    // =========================================================================

    // -------------------------------------------------------------------------
    // L-0: Loss of human life or crew injury
    // SOURCE: STPA-BPS-001 §2; SOLAS Chapter III
    // -------------------------------------------------------------------------
    #Loss {
        lossId = "L-0";
        description = "Loss of human life or injury to vessel crew";
        category = "safety";
        severity = "catastrophic";
        regulatoryRef = "SOLAS Chapter III — Life-Saving Appliances";
        sourceDoc = "STPA-BPS-001";
        section = "2.1";
    }
    attribute def Loss_L0 {}

    // -------------------------------------------------------------------------
    // L-1: Loss of vessel — sinking or capsize due to uncontrolled flooding
    // SOURCE: STPA-BPS-001 §2; SOLAS Chapter II-1
    // -------------------------------------------------------------------------
    #Loss {
        lossId = "L-1";
        description = "Loss of vessel — sinking or capsize due to uncontrolled flooding";
        category = "safety";
        severity = "catastrophic";
        regulatoryRef = "SOLAS Chapter II-1 — Subdivision and Stability";
        sourceDoc = "STPA-BPS-001";
        section = "2.2";
    }
    attribute def Loss_L1 {}

    // -------------------------------------------------------------------------
    // L-2: Marine environmental damage
    // SOURCE: STPA-BPS-001 §2; MARPOL 73/78 Annex I Reg.22
    // -------------------------------------------------------------------------
    #Loss {
        lossId = "L-2";
        description = "Marine environmental damage — bilge water overboard without treatment";
        category = "environmental";
        severity = "critical";
        regulatoryRef = "MARPOL 73/78 Annex I, Regulation 22";
        sourceDoc = "STPA-BPS-001";
        section = "2.3";
    }
    attribute def Loss_L2 {}

    // =========================================================================
    // UCA-DERIVED SAFETY REQUIREMENTS
    //
    // Each requirement def below was derived from an Unsafe Control Action
    // identified during STPA analysis. The require constraint body formalises
    // the STPA safety constraint that eliminates the UCA.
    //
    // Pattern:
    // requirement def <UCA_SafetyRequirementName> {
    // #UCA { ... }
    // #SafetyRequirement { ... }
    // subject sys : BilgePumpSystem;
    // require constraint { <UCA elimination condition> }
    // }
    // =========================================================================

    // -------------------------------------------------------------------------
    // UCA-001 Safety Requirement: Controller must activate Pump A
    // when water level exceeds trigger threshold in AUTO mode.
    // DERIVED FROM: UCA-001 "ActivatePumpA — Not Provided"
    // SOURCE: STPA-BPS-002 §3.1
    // -------------------------------------------------------------------------
    #UCA {
        ucaId = "UCA-001";
        controlAction = "ActivatePumpA";
        guideword = "Not Provided";
        context = "Water level ≥ triggerLevel_m in AUTO mode";
        hazardRefs = "H-1,HS-1";
        severity = "catastrophic";
        failureModeLink = "FM-C-001";
        transitionRef = "MONITORING_to_PUMP_A_ACTIVE";
        sourceDoc = "STPA-BPS-002";
        section = "3.1";
    }
    #SafetyRequirement {
        srId = "SR-001";
        derivedFrom = "UCA-001";
        rationale =
            "Eliminates scenario where flooding proceeds unchecked because controller never issues pump start command";
        verificationMethod = "analysis";
        sourceDoc = "STPA-BPS-002";
        section = "5.1";
    }
    requirement def UCA_001_ControllerNoActivatePumpA {
        subject sys : BilgePumpSystem;

        doc
        /* The PumpController shall issue an ActivatePumpA command within
         * the designed response window whenever sensor.waterLevel exceeds
         * triggerLevel_m in automatic operating mode.
         * Derived from STPA UCA-001; eliminates H-1 (bilge flooding).
         */

        require constraint {
            sys.controller.responseTime_s <= 5.0
        }
    }

    // -------------------------------------------------------------------------
    // UCA-002 Safety Requirement: Sensor must provide valid water level readings.
    // A fail-silent (stuck-at-zero) sensor fault is a hazardous UCA.
    // DERIVED FROM: UCA-002 "ReportWaterLevel — Wrong Value Provided"
    // SOURCE: STPA-BPS-002 §3.2
    // -------------------------------------------------------------------------
    #UCA {
        ucaId = "UCA-002";
        controlAction = "ReportWaterLevel";
        guideword = "Wrong Value Provided";
        context = "Sensor stuck-at-zero during actual bilge flooding";
        hazardRefs = "H-1,HS-1";
        severity = "catastrophic";
        failureModeLink = "FM-S-001";
        sourceDoc = "STPA-BPS-002";
        section = "3.2";
    }
    #SafetyRequirement {
        srId = "SR-002";
        derivedFrom = "UCA-002";
        rationale =
            "Eliminates fail-silent sensor fault by requiring readings stay within plausible operational range";
        verificationMethod = "analysis";
        sourceDoc = "STPA-BPS-002";
        section = "5.2";
    }
    requirement def UCA_002_SensorFailSilent {
        subject sys : BilgePumpSystem;

        doc
        /* The BilgeWaterSensor shall report a water level value within the
         * valid operational range at all times. A reading of exactly 0.0 m
         * when pumps are inactive is indicative of a fail-silent fault.
         * Derived from STPA UCA-002; eliminates H-1.
         */

        require constraint {
            sys.sensor.waterLevel >= 0.0 and sys.sensor.waterLevel <= 1.0
        }
    }

    // -------------------------------------------------------------------------
    // UCA-003 Safety Requirement: Controller must trigger alarm on flood detection.
    // DERIVED FROM: UCA-003 "TriggerAlarm — Not Provided"
    // SOURCE: STPA-BPS-002 §3.3
    // -------------------------------------------------------------------------
    #UCA {
        ucaId = "UCA-003";
        controlAction = "TriggerAlarm";
        guideword = "Not Provided";
        context = "Water level > 0.25 m detected; controller does not assert alarmOut";
        hazardRefs = "H-3,HS-2";
        severity = "critical";
        failureModeLink = "FM-C-002";
        transitionRef = "PUMP_A_ACTIVE_to_ALARM_TRIGGERED";
        sourceDoc = "STPA-BPS-002";
        section = "3.3";
    }
    #SafetyRequirement {
        srId = "SR-003";
        derivedFrom = "UCA-003";
        rationale = "Eliminates alarm suppression scenario; crew must receive timely notification";
        verificationMethod = "analysis";
        sourceDoc = "STPA-BPS-002";
        section = "5.3";
    }
    requirement def UCA_003_ControllerNoAlarm {
        subject sys : BilgePumpSystem;

        doc
        /* When water level exceeds the trigger threshold, the AlarmSystem
         * must receive the trigger within the IEC 60945 §4.3 response
         * window. The controller shall not suppress alarm output.
         * Derived from STPA UCA-003; eliminates H-3.
         */

        require constraint {
            sys.alarm.activationDelay_s <= 2.0
        }
    }

    // -------------------------------------------------------------------------
    // UCA-004 Safety Requirement: Controller must activate Pump B when Pump A fails.
    // DERIVED FROM: UCA-004 "ActivatePumpB — Not Provided (on Pump A fault)"
    // SOURCE: STPA-BPS-002 §3.4
    // -------------------------------------------------------------------------
    #UCA {
        ucaId = "UCA-004";
        controlAction = "ActivatePumpB";
        guideword = "Not Provided";
        context = "Pump A faulted (zero flow); controller does not switch to Pump B";
        hazardRefs = "H-4,HS-3";
        severity = "critical";
        failureModeLink = "FM-C-003";
        transitionRef = "PUMP_A_ACTIVE_to_FAILOVER";
        sourceDoc = "STPA-BPS-002";
        section = "3.4";
    }
    #SafetyRequirement {
        srId = "SR-004";
        derivedFrom = "UCA-004";
        rationale =
            "Eliminates single-pump failure scenario by requiring active failover to redundant pump";
        verificationMethod = "analysis";
        sourceDoc = "STPA-BPS-002";
        section = "5.4";
    }
    requirement def UCA_004_ControllerNoFailover {
        subject sys : BilgePumpSystem;

        doc
        /* The PumpController shall maintain the isRedundant designation on
         * Pump B, ensuring the redundant pump is available for automatic
         * failover. The failover path must be structurally enabled.
         * Derived from STPA UCA-004; eliminates H-4.
         */

        require constraint {
            sys.pumpB.isRedundant == true
        }
    }

    // -------------------------------------------------------------------------
    // UCA-005 Safety Requirement: Pump A activation must not be delayed.
    // DERIVED FROM: UCA-005 "ActivatePumpA — Provided Too Late"
    // SOURCE: STPA-BPS-002 §3.5
    // -------------------------------------------------------------------------
    #UCA {
        ucaId = "UCA-005";
        controlAction = "ActivatePumpA";
        guideword = "Provided Too Late";
        context = "Pump A command issued > 5 s after water level crosses trigger threshold";
        hazardRefs = "H-1";
        severity = "critical";
        failureModeLink = "FM-C-001";
        transitionRef = "MONITORING_to_PUMP_A_ACTIVE";
        sourceDoc = "STPA-BPS-002";
        section = "3.5";
    }
    #SafetyRequirement {
        srId = "SR-005";
        derivedFrom = "UCA-005";
        rationale =
            "Controller response time must be bounded to prevent flooding accumulation during delay";
        verificationMethod = "analysis";
        sourceDoc = "STPA-BPS-002";
        section = "5.5";
    }
    requirement def UCA_005_ControllerDelayedActivation {
        subject sys : BilgePumpSystem;

        doc
        /* The PumpController response time (from trigger-level detection
         * to pump command issuance) shall not exceed 5.0 seconds.
         * Derived from STPA UCA-005; eliminates late-activation variant of H-1.
         */

        require constraint {
            sys.controller.responseTime_s <= 5.0
        }
    }
} // package BilgePump_Safety


Package BilgePump_Safety (a10130d9-f8c4-4882-be05-4d92fd3bde12)


In [6]:
// =============================================================================
// BilgePump::FMEA
// Layer: Failure Mode and Effects Analysis
// Purpose: FMEA artifacts for the bilge pump system -- failure-mode attribute
// and constraint definitions (RPN, parallel failure rate, NPSH,
// failover timing) plus NEGATIVE verification cases that inject
// failure-mode values and confirm the affected requirements VIOLATE.
//
// This layer is parallel to Safety.sysml; both are imported into the unified
// BilgePump project for the complete verification picture.
//
// KERNEL-EVALUABLE NEGATIVE TESTS
// -------------------------------
// The kernel cannot override an attribute value already bound by Architecture
// (e.g. pumpA.flowRate, alarm.activationDelay_s).  Each fault is therefore
// injected on a fresh component instance typed by its Library part def (whose
// attributes are unbound), and the affected requirement is re-expressed as a
// computed Boolean.  Every Boolean below is expected to evaluate to FALSE
// (i.e. VIOLATED), which the publish pipeline captures via %eval and records in
// the `sysml_assertions` table with result_bool = false.
//
// %eval BilgePump_FMEA::FM_S_001_alarm     -> LiteralBoolean false   (VIOLATED)
//
// SOURCE DOCUMENTS:
// FMEA-BPS-001 -- FMEA table (failure modes; S/O/D ratings)
// FMEA-BPS-002 -- Reliability and risk constraint equations
// FMEA-BPS-003 -- Negative test scenarios (one per critical failure mode)
// =============================================================================

package BilgePump_FMEA {
    private import ScalarValues::*;
    // RAAML stereotypes (Hazard/Loss/UCA/FailureMode/…) were folded into
    // Requirements.sysml; they arrive via the BilgePump_Requirements import below.
    public import BilgePump_Library::*;
    public import BilgePump_Architecture::*;
    public import BilgePump_Requirements::*;

    // =========================================================================
    // Failure Mode Attribute Definition
    // Extends Library attribute types with FMEA rating fields.
    // SOURCE: FMEA-BPS-001; MIL-STD-1629A; AIAG FMEA 4th Edition
    // NOTE: 'occurrence' is a reserved SysML v2 keyword -> 'occurrenceRating'.
    // =========================================================================
    attribute def FailureModeAttr {
        attribute severity : Real; // 1-10 AIAG severity scale
        attribute occurrenceRating : Real; // 1-10 occurrence likelihood
        attribute detection : Real; // 1-10 detection capability
        attribute rpn : Real; // Risk Priority Number = S x O x D
    }

    // =========================================================================
    // FMEA CONSTRAINT DEFINITIONS  (from FMEA-BPS-002)
    // =========================================================================

    // RiskPriorityNumber -- RPN = S x O x D.
    // SOURCE: FMEA-BPS-002 §2.1; MIL-STD-1629A; AIAG FMEA 4th Edition.
    // Fault-tree tag: FT-RPN, top event "High Risk Priority Number (RPN >= 100)".
    constraint def RiskPriorityNumber {
        attribute severity : Real; // Input: S rating (1-10)
        attribute occurrenceRating : Real; // Input: O rating (1-10)
        attribute detection : Real; // Input: D rating (1-10)
        attribute rpn : Real; // Output: computed RPN

        assert constraint {
            rpn == severity * occurrenceRating * detection
        }
    }

    // ParallelRedundancyFailureRate -- system failure rate for two pumps in
    // active parallel redundancy (both must fail; exponential model).
    // SOURCE: FMEA-BPS-002 §2.2; IEC 61508-6 Table B.5; MIL-HDBK-217F.
    constraint def ParallelRedundancyFailureRate {
        attribute lambda_A : Real; // Pump A failure rate (failures/hour)
        attribute lambda_B : Real; // Pump B failure rate (= lambda_A, identical pump)
        attribute lambda_sys : Real; // System failure rate with parallel redundancy

        assert constraint {
            lambda_sys == lambda_A * lambda_B
        }
    }

    // NPSHMarginCheck -- cavitation avoidance: available NPSH must exceed
    // required NPSH by the Hydraulic Institute design margin (>= 1.0 m).
    // SOURCE: FMEA-BPS-002 §2.3; Hydraulic Institute Standards; ISO 9906.
    // FAILURE MODE LINK: FM-PA-002 (Pump A cavitation).
    constraint def NPSHMarginCheck {
        attribute npsh_available : Real; // NPSH available (m) from suction geometry
        attribute npsh_required : Real; // NPSH required at rated flow (CFD curve)
        attribute npsh_margin : Real; // Design margin (Hydraulic Institute >= 1.0 m)

        assert constraint {
            npsh_available >= npsh_required + npsh_margin
        }
    }

    // FailoverSwitchTimeConstraint -- timing of controller failover A -> B.
    // SOURCE: SIM-CTRL-001 §3.2. FAILURE MODE LINK: FM-C-003.
    constraint def FailoverSwitchTimeConstraint {
        attribute failoverTime_s : Real; // Actual fault-to-Pump-B-command time (s)
        attribute maxFailoverTime_s : Real; // Maximum allowed failover time (s)

        assert constraint {
            failoverTime_s <= maxFailoverTime_s
        }
    }

    // =========================================================================
    // FMEA NEGATIVE TEST CASES (fault injection -> expected VIOLATED)
    //
    // Each fault is injected on Library-typed component instances; the affected
    // requirement is re-expressed as a computed Boolean expected to be FALSE.
    // RPN action threshold: 100 (FMEA-BPS-001).
    // =========================================================================

    // -------------------------------------------------------------------------
    // FM-S-001: BilgeWaterSensor stuck-at-zero (fail-silent).  RPN=240.
    // SOURCE: FMEA-BPS-003 §3.1.
    // Effect: controller never activates pumps; alarm never fires.
    // EXPECTED VIOLATED: AlarmResponseRequirement, DischargeCapacityRequirement.
    // -------------------------------------------------------------------------
    part fmS001_alarm : AlarmSystem {
        @FailureMode {
            fmId = "FM-S-001";
            component = "BilgeWaterSensor";
            instance = "sensor";
            failureModeText = "Stuck-at-zero output (fail-silent)";
            failureEffect = "Controller never activates pumps; bilge floods unchecked";
            severity = 10;
            occurrenceRating = 3;
            detection = 8;
            rpn = 240;
            ucaRef = "UCA-002";
            hazardRef = "H-1";
            sourceDoc = "FMEA-BPS-001";
            section = "3.1";
        }
        attribute :>> activationDelay_s = 999.0; // FAULT: alarm never annunciates
    }
    part fmS001_pumpA : BilgePumpA { attribute :>> flowRate = 0.0; } // not activated
    part fmS001_pumpB : BilgePumpB { attribute :>> flowRate = 0.0; } // not activated
    // VIOLATED: alarm delay 999 s > 2.0 s [BPS-REQ-003]
    attribute FM_S_001_alarm : Boolean = fmS001_alarm.activationDelay_s <= 2.0;
    // VIOLATED: combined flow 0 < 0.030 m^3/s [BPS-REQ-004]
    attribute FM_S_001_discharge : Boolean = fmS001_pumpA.flowRate + fmS001_pumpB.flowRate >= 0.030;

    // -------------------------------------------------------------------------
    // FM-PA-002: BilgePumpA cavitation (degraded efficiency 0.40).  RPN=210.
    // SOURCE: FMEA-BPS-003 §3.2.
    // EXPECTED VIOLATED: EffectiveDischargeCapacityRequirement (per-pump eta).
    // netFlow = (0.025*0.40 + 0.025*0.82) * (1-0.05) = 0.028975 < 0.030.
    // Note: the raw-sum DischargeCapacityRequirement would NOT catch this; the
    // effective (efficiency-aware) check FT-002 is the discriminating test.
    // -------------------------------------------------------------------------
    part fmPA002_pumpA : BilgePumpA {
        @FailureMode {
            fmId = "FM-PA-002";
            component = "BilgePumpA";
            instance = "pumpA";
            failureModeText = "Reduced efficiency due to cavitation at low NPSH";
            failureEffect = "Net discharge flow drops below design inflow threshold";
            severity = 7;
            occurrenceRating = 5;
            detection = 6;
            rpn = 210;
            ucaRef = "";
            hazardRef = "H-1";
            sourceDoc = "FMEA-BPS-001";
            section = "4.2";
        }
        attribute :>> flowRate = 0.025;
        attribute :>> efficiency = 0.40; // FAULT: cavitation-degraded efficiency
    }
    part fmPA002_pumpB : BilgePumpB {
        attribute :>> flowRate = 0.025;
        attribute :>> efficiency = 0.82; // Pump B unaffected
    }
    part fmPA002_disch : DischargeLine { attribute :>> pipeLossFactor = 0.05; }
    attribute FM_PA_002_netFlow : Real =
        (fmPA002_pumpA.flowRate * fmPA002_pumpA.efficiency +
            fmPA002_pumpB.flowRate * fmPA002_pumpB.efficiency) *
        (1.0 - fmPA002_disch.pipeLossFactor);
    // VIOLATED: netFlow 0.028975 < 0.030 [BPS-FT-002]
    attribute FM_PA_002_effDischarge : Boolean = FM_PA_002_netFlow >= 0.030;

    // -------------------------------------------------------------------------
    // FM-PB-001: BilgePumpB redundant feed lost (isRedundant -> false).  RPN=72
    // (Severity=9 flags mandatory review).  SOURCE: FMEA-BPS-003 §3.3.
    // EXPECTED VIOLATED: PumpRedundancyRequirement.
    // -------------------------------------------------------------------------
    part fmPB001_pumpB : BilgePumpB {
        @FailureMode {
            fmId = "FM-PB-001";
            component = "BilgePumpB";
            instance = "pumpB";
            failureModeText = "Redundant pump unavailable -- independent power bus lost";
            failureEffect = "PumpB.isRedundant effectively false; single point of failure";
            severity = 9;
            occurrenceRating = 2;
            detection = 4;
            rpn = 72;
            ucaRef = "";
            hazardRef = "H-4";
            sourceDoc = "FMEA-BPS-001";
            section = "5.1";
        }
        attribute :>> isRedundant = false; // FAULT: power feed not independent
    }
    // VIOLATED: isRedundant == true is false [BPS-REQ-002]
    attribute FM_PB_001_redundancy : Boolean = fmPB001_pumpB.isRedundant == true;

    // -------------------------------------------------------------------------
    // FM-C-001: PumpController software hang (no outputs).  RPN=140.
    // SOURCE: FMEA-BPS-003 §3.4.  State: FAULT.
    // EXPECTED VIOLATED: ControllerActivationTimingRequirement,
    // DischargeCapacityRequirement, AlarmResponseRequirement.
    // -------------------------------------------------------------------------
    part fmC001_controller : PumpController {
        @FailureMode {
            fmId = "FM-C-001";
            component = "PumpController";
            instance = "controller";
            failureModeText = "Controller software hang -- no outputs issued";
            failureEffect = "No pump commands; no alarm; all automation paths fail";
            severity = 10;
            occurrenceRating = 2;
            detection = 7;
            rpn = 140;
            ucaRef = "UCA-001";
            hazardRef = "H-1";
            stateRef = "FAULT";
            sourceDoc = "FMEA-BPS-001";
            section = "6.1";
        }
        attribute :>> responseTime_s = 9999.0; // FAULT: controller hung
    }
    part fmC001_alarm : AlarmSystem { attribute :>> activationDelay_s = 9999.0; } // output frozen
    part fmC001_pumpA : BilgePumpA { attribute :>> flowRate = 0.0; }
    part fmC001_pumpB : BilgePumpB { attribute :>> flowRate = 0.0; }
    // VIOLATED: responseTime 9999 > 5.0 s [BPS-REQ-005]
    attribute FM_C_001_activation : Boolean = fmC001_controller.responseTime_s <= 5.0;
    // VIOLATED: alarm delay 9999 > 2.0 s [BPS-REQ-003]
    attribute FM_C_001_alarm : Boolean = fmC001_alarm.activationDelay_s <= 2.0;
    // VIOLATED: combined flow 0 < 0.030 m^3/s [BPS-REQ-004]
    attribute FM_C_001_discharge : Boolean = fmC001_pumpA.flowRate + fmC001_pumpB.flowRate >= 0.030;

    // -------------------------------------------------------------------------
    // FM-C-003: PumpController failover not triggered (failoverTime 9999).
    // RPN=90 (Severity=9 flags mandatory review).  State: FAILOVER.
    // SOURCE: FMEA-BPS-001 §6.3; SIM-CTRL-001 §3.2 (no-failover scenario).
    // GAP CLOSED: BPS-REQ-002 verified structural redundancy but not failover
    // TIMING; this closes it via BPS-REQ-006.
    // EXPECTED VIOLATED: FailoverSwitchTimingRequirement, DischargeCapacityRequirement.
    // -------------------------------------------------------------------------
    part fmC003_controller : PumpController {
        @FailureMode {
            fmId = "FM-C-003";
            component = "PumpController";
            instance = "controller";
            failureModeText = "Failover path not triggered -- Pump B command never issued";
            failureEffect = "Pump A fault uncompensated; combined discharge drops to zero";
            severity = 9;
            occurrenceRating = 2;
            detection = 5;
            rpn = 90;
            ucaRef = "UCA-004";
            hazardRef = "H-4";
            stateRef = "FAILOVER";
            sourceDoc = "FMEA-BPS-001";
            section = "6.3";
        }
        attribute :>> failoverTime_s = 9999.0; // FAULT: failover command never issued
    }
    part fmC003_pumpA : BilgePumpA { attribute :>> flowRate = 0.0; } // Pump A failed
    part fmC003_pumpB : BilgePumpB { attribute :>> flowRate = 0.0; } // failover not triggered
    // VIOLATED: failoverTime 9999 > 3.0 s [BPS-REQ-006]
    attribute FM_C_003_failover : Boolean = fmC003_controller.failoverTime_s <= 3.0;
    // VIOLATED: combined flow 0 < 0.030 m^3/s [BPS-REQ-004]
    attribute FM_C_003_discharge : Boolean = fmC003_pumpA.flowRate + fmC003_pumpB.flowRate >= 0.030;
} // package BilgePump_FMEA


Package BilgePump_FMEA (c4d626df-323c-4cba-bf34-403bd94e34ac)


In [7]:
// =============================================================================
// BilgePump::StateMachine
// Layer: Behavioral Modeling
// Purpose: Discrete-event state machine definitions for the BilgePumpSystem.
// Two interlocking state machines:
// 1. PumpControllerBehavior — 7-state sub-machine for PumpController
// 2. BilgePumpSystemBehavior — 5-state top-level system machine
// (aggregates controller sub-states into crew-visible modes)
//
// Import chain:
// BilgePump::Library       — part def and attribute types
// BilgePump::Architecture  — BilgePumpSystem, PumpController part instances
//
// SOURCE DOCUMENTS:
// SIM-CTRL-001  — Simulink/Stateflow discrete-event timing model (controller firmware)
// STPA-BPS-002  — Unsafe Control Actions (UCA guidewords → transition semantics)
// FMEA-BPS-001  — Failure mode → fault/failover state mapping
//
// STATE DESIGN RATIONALE:
// (a) SIM-CTRL-001 timing bounds: responseTime_s ≤ 5.0 s, failoverTime_s ≤ 3.0 s
// (b) UCA guidewords "Not Provided" and "Provided Too Late" define activation transitions
// (c) FM-C-001 (controller hang) → FAULT state
// FM-C-003 (failover not triggered) → FAILOVER state
//
// INTEGRATION WITH OTHER LAYERS:
// Architecture.sysml : exhibit statements link part instances to state defs here
// Requirements.sysml : BPS-REQ-005, BPS-REQ-006 — timing guards formalised
// Analysis.sysml     : BilgePumpTimingVerification asserts BPS-REQ-005/006
// Safety.sysml       : transitionRef fields in #UCA blocks cross-reference transitions
// FMEA.sysml         : stateRef fields in #FailureMode blocks cross-reference states
//
// STATE SPACE RESPONSIBILITY:
// The agent output artifact lib/state-space.json maps each state to its owning
// component, entry/exit conditions, linked safety requirements (SR-xxx),
// linked failure modes (FM-xxx), and timing constraints.
// See sysml-state-machine-mapper.agent.md for the generation specification.
//
// PILOT API NOTE:
// state def and transition are core SysML v2 constructs (SysML v2 §7.14).
// Guard expressions use inline comments rather than executable if-clauses to
// ensure compatibility across Pilot API JAR versions. The guard semantics are
// formally captured in linked constraint defs (Analysis.sysml) and requirement
// defs (Requirements.sysml).
// =============================================================================

package BilgePump_StateMachine {
    private import ScalarValues::*;
    public import BilgePump_Library::*;
    public import BilgePump_Architecture::*;

    // =========================================================================
    // Requirement→state trace (Epic-1 extensibility prototype — step 3)
    //
    // Trace the two genuinely temporal/behavioral DNV requirements to the states
    // that ALREADY exist (no new states, no new behaviour):
    //   #11 (§8.2.2 failover / degraded mode — the other unit makes up a
    //        capacity deficiency)  → existing FAILOVER state
    //   #21 (§8.3.3 concurrency — direct suctions usable simultaneously with the
    //        other unit)           → existing DUAL_PUMP_ACTIVE state
    //
    // The Pilot kernel REJECTS `satisfy <RequirementDef> by <Type>;` and
    // `verify <RequirementDef> by <AnalysisDef>;` (they need a requirement usage
    // + a feature, not defs — see Analysis.sysml). This model already links to
    // states via metadata attributes (RAAML FailureMode.stateRef, UCA.transitionRef).
    // We follow that proven idiom: a standalone metadata usage carrying reqRef +
    // stateRef (same shape as the #Loss usages that publish in the 9/9 run).
    // These land in the MetadataUsage table, queryable by reqRef / stateRef.
    // Guards stay comments (executable temporal guards are Epic-2).
    // =========================================================================
    metadata def RequirementStateTrace {
        attribute reqRef   : String; // requirement id (DNV corpus shortlist #)
        attribute stateRef : String; // existing state name traced to
        attribute section  : String; // DNV RU-SHIP Pt.4 Ch.6 Sec.8 subsection
        attribute rationale : String;
    }

    #RequirementStateTrace {
        reqRef    = "REQ-11";
        stateRef  = "FAILOVER";
        section   = "8.2.2";
        rationale = "failover / degraded mode — the other unit makes up a capacity deficiency";
    }
    #RequirementStateTrace {
        reqRef    = "REQ-21";
        stateRef  = "DUAL_PUMP_ACTIVE";
        section   = "8.3.3";
        rationale = "concurrency — direct suctions usable simultaneously with the other unit";
    }

    // =========================================================================
    // PumpControllerBehavior
    // Sub-state machine for the PumpController component.
    // Exhibited by: controller : PumpController (Architecture.sysml)
    //
    // STATE INVENTORY (7 states):
    // IDLE              — system energised, no bilge threat detected
    // MONITORING        — continuously polling sensor.waterLevel
    // PUMP_A_ACTIVE     — Pump A commanded ON; waterLevel ≥ triggerLevel_m
    // DUAL_PUMP_ACTIVE  — both pumps active (high-demand or failover handoff complete)
    // ALARM_TRIGGERED   — alarm.alarmIn asserted; IEC 60945 notification window active
    // FAILOVER          — Pump A fault detected; Pump B bearing sole responsibility
    // FAULT             — unrecoverable state; controller hang or both pumps unavailable
    //
    // TIMING CONSTRAINTS (source: SIM-CTRL-001):
    // MONITORING → PUMP_A_ACTIVE    : responseTime_s ≤ 5.0 s  [UCA-001/005, BPS-REQ-005]
    // PUMP_A_ACTIVE → ALARM_TRIGGERED: activationDelay_s ≤ 2.0 s [UCA-003, BPS-REQ-003]
    // PUMP_A_ACTIVE → FAILOVER       : failoverTime_s ≤ 3.0 s   [UCA-004, BPS-REQ-006]
    // =========================================================================
    state def PumpControllerBehavior {
        // -----------------------------------------------------------------
        // IDLE
        // Entry condition:  system power-on or operator hardware reset via ui.overrideOut
        // Exit condition:   auto-transition to MONITORING on startup
        // Linked SRs:       none (passive, no constraints active)
        // Linked FMs:       none
        // -----------------------------------------------------------------
        state IDLE;

        // -----------------------------------------------------------------
        // MONITORING
        // Entry condition:  auto-transition from IDLE; re-entered after ALARM ack
        // or after dewatering completes (waterLevel < triggerLevel_m)
        // Do:               continuous evaluation of sensor.waterLevel vs triggerLevel_m
        // Exit condition:   waterLevel ≥ triggerLevel_m → PUMP_A_ACTIVE
        // Timing guard:     responseTime_s ≤ 5.0 s from detection to command [BPS-REQ-005]
        // SOURCE: SIM-CTRL-001 §3.1 (nominal 1.0 s; watchdog 4.9 s)
        // -----------------------------------------------------------------
        state MONITORING;

        // -----------------------------------------------------------------
        // PUMP_A_ACTIVE
        // Entry condition:  waterLevel ≥ triggerLevel_m; responseTime_s ≤ 5.0 s [SR-001/005]
        // Exit paths:
        // → ALARM_TRIGGERED   : activationDelay_s ≤ 2.0 s [SR-003, BPS-REQ-003]
        // → FAILOVER          : Pump A fault; failoverTime_s ≤ 3.0 s [SR-004, BPS-REQ-006]
        // → MONITORING        : waterLevel < triggerLevel_m (dewatering complete)
        // → FAULT             : responseTime_s > 5.0 s (controller hang, FM-C-001)
        // -----------------------------------------------------------------
        state PUMP_A_ACTIVE;

        // -----------------------------------------------------------------
        // DUAL_PUMP_ACTIVE
        // Entry condition:  (a) high bilge inflow demands both pumps, OR
        // (b) FAILOVER handoff complete — Pump B at full capacity
        // Exit condition:   waterLevel < triggerLevel_m → MONITORING
        // NOTE: Q_total = pumpA.flowRate + pumpB.flowRate; satisfies BPS-REQ-004
        // -----------------------------------------------------------------
        state DUAL_PUMP_ACTIVE;

        // -----------------------------------------------------------------
        // ALARM_TRIGGERED
        // Entry condition:  waterLevel > alarm threshold AND activationDelay_s ≤ 2.0 s
        // SOURCE: IEC 60945 §4.3; SIM-CTRL-001 alarm logic
        // Linked: UCA-003 (transitionRef), SR-003, BPS-REQ-003
        // Exit condition:   operator acknowledgement via ui.overrideOut → MONITORING
        // -----------------------------------------------------------------
        state ALARM_TRIGGERED;

        // -----------------------------------------------------------------
        // FAILOVER
        // Entry condition:  Pump A fault detected; controller issues ActivatePumpB command
        // failoverTime_s ≤ 3.0 s from fault detection to Pump B command
        // SOURCE: SIM-CTRL-001 §3.2 (nominal 0.8 s; delayed 2.7 s)
        // Linked: UCA-004 (transitionRef), SR-004, BPS-REQ-006
        // Exit condition:   Pump B reaches operational flow → DUAL_PUMP_ACTIVE
        // Linked FMs:       FM-C-003 (failover not triggered — violates BPS-REQ-006)
        // -----------------------------------------------------------------
        state FAILOVER;

        // -----------------------------------------------------------------
        // FAULT
        // Entry condition:  controller hang (FM-C-001) OR both pumps unavailable
        // Violates BPS-REQ-005 (responseTime_s → effectively ∞)
        // Exit condition:   operator hardware reset → IDLE (requires manual intervention)
        // Linked FMs:       FM-C-001 (RPN=140, S=10, O=2, D=7)
        // DESIGN NOTE:      SIM-CTRL-001 §3.1 — hardware watchdog recommended to prevent
        // software hang from being unrecoverable
        // -----------------------------------------------------------------
        state FAULT;

        // -----------------------------------------------------------------
        // TRANSITIONS
        // Names are the canonical cross-reference identifiers used in:
        // Safety.sysml  #UCA blocks → transitionRef field
        // lib/state-space.json     → coverage.uca_coverage entries
        // -----------------------------------------------------------------

        // Startup: enter IDLE first
        entry;
        then IDLE;

        // IDLE → MONITORING (auto-transition on system startup)
        // Condition: system power on; no active fault
        transition IDLE_to_MONITORING first IDLE then MONITORING;

        // MONITORING → PUMP_A_ACTIVE
        // Trigger:  waterLevel crosses triggerLevel_m (sensor.levelOut > controller threshold)
        // Timing:   responseTime_s ≤ 5.0 s from detection to pump command [UCA-001/005, BPS-REQ-005]
        // SOURCE:   SIM-CTRL-001 §3.1; STPA UCA-001 "ActivatePumpA — Not Provided"
        // transitionRef used in Safety.sysml: UCA-001, UCA-005
        // guard: sys.sensor.waterLevel >= sys.controller.triggerLevel_m
        // AND sys.controller.responseTime_s <= 5.0
        transition MONITORING_to_PUMP_A_ACTIVE first MONITORING then PUMP_A_ACTIVE;

        // PUMP_A_ACTIVE → ALARM_TRIGGERED
        // Trigger:  water level continues rising above alarm threshold
        // Timing:   activationDelay_s ≤ 2.0 s from trigger to alarm annunciation [UCA-003]
        // SOURCE: IEC 60945 §4.3; STPA UCA-003 "TriggerAlarm — Not Provided"
        // transitionRef used in Safety.sysml: UCA-003
        // guard: sys.alarm.activationDelay_s <= 2.0
        transition PUMP_A_ACTIVE_to_ALARM_TRIGGERED first PUMP_A_ACTIVE then ALARM_TRIGGERED;

        // PUMP_A_ACTIVE → FAILOVER
        // Trigger:  Pump A fault detected (zero flow or drive fault signal)
        // Timing:   failoverTime_s ≤ 3.0 s from fault detection to Pump B command [UCA-004, BPS-REQ-006]
        // SOURCE: SIM-CTRL-001 §3.2; STPA UCA-004 "ActivatePumpB — Not Provided"
        // transitionRef used in Safety.sysml: UCA-004
        // guard: sys.pumpA.flowRate == 0.0 AND sys.controller.failoverTime_s <= 3.0
        transition PUMP_A_ACTIVE_to_FAILOVER first PUMP_A_ACTIVE then FAILOVER;

        // PUMP_A_ACTIVE → FAULT (controller hang; FM-C-001)
        // Trigger:  controller responseTime_s exceeds maximum bound
        // NOTE:     In practice caught by hardware watchdog; modeled for FMEA traceability
        // guard: sys.controller.responseTime_s > 5.0
        transition PUMP_A_ACTIVE_to_FAULT first PUMP_A_ACTIVE then FAULT;

        // PUMP_A_ACTIVE → MONITORING (normal dewatering complete)
        // Trigger:  waterLevel drops below triggerLevel_m after pumping
        // guard: sys.sensor.waterLevel < sys.controller.triggerLevel_m
        transition PUMP_A_ACTIVE_to_MONITORING first PUMP_A_ACTIVE then MONITORING;

        // FAILOVER → DUAL_PUMP_ACTIVE
        // Trigger:  Pump B reaches operational flow rate; Pump A still faulted
        // guard: sys.pumpB.flowRate > 0.0
        transition FAILOVER_to_DUAL_PUMP_ACTIVE first FAILOVER then DUAL_PUMP_ACTIVE;

        // DUAL_PUMP_ACTIVE → MONITORING (dewatering complete; both pumps can stop)
        // guard: sys.sensor.waterLevel < sys.controller.triggerLevel_m
        transition DUAL_PUMP_ACTIVE_to_MONITORING first DUAL_PUMP_ACTIVE then MONITORING;

        // ALARM_TRIGGERED → MONITORING (operator acknowledges alarm)
        // Trigger:  operator action via ui.overrideOut port (bridge panel or HMI)
        // guard: sys.ui.overrideActive == true
        transition ALARM_TRIGGERED_to_MONITORING first ALARM_TRIGGERED then MONITORING;

        // FAULT → IDLE (operator hardware reset)
        // NOTE: Requires physical hardware action; cannot be cleared by software alone
        // guard: sys.ui.overrideActive == true (hardware reset acknowledged)
        transition FAULT_to_IDLE first FAULT then IDLE;
    }

    // =========================================================================
    // BilgePumpSystemBehavior
    // Top-level system state machine. Aggregates PumpController sub-states into
    // 5 system-level operational modes visible to crew and SCADA.
    // Exhibited by: sys : BilgePumpSystem (Architecture.sysml)
    //
    // STATE INVENTORY (5 states):
    // STANDBY         — normal operation; no bilge threat
    // PUMPING         — active dewatering underway
    // ALERT           — alarm active; awaiting crew acknowledgement
    // FAILOVER_ACTIVE — Pump A faulted; Pump B bearing sole load
    // CRITICAL_FAULT  — controller or dual-pump failure; SOLAS emergency protocol
    //
    // CREW-VISIBLE STATUS (maps to StatusPort.status string on controller.statusOut):
    // STANDBY        → "NORMAL"
    // PUMPING        → "PUMPING"
    // ALERT          → "FAULT"    (alarm annunciation active)
    // FAILOVER_ACTIVE → "FAULT"  (reduced redundancy — Pump A unavailable)
    // CRITICAL_FAULT → "FAULT"   + physical alarm latched (manual reset required)
    //
    // CONTROLLER SUB-STATE CORRESPONDENCE:
    // STANDBY        ← controller in IDLE or MONITORING
    // PUMPING        ← controller in PUMP_A_ACTIVE or DUAL_PUMP_ACTIVE
    // ALERT          ← controller in ALARM_TRIGGERED
    // FAILOVER_ACTIVE← controller in FAILOVER
    // CRITICAL_FAULT ← controller in FAULT
    // =========================================================================
    state def BilgePumpSystemBehavior {
        // STANDBY — normal operations; sensor polling; no active dewatering
        // Linked: no active requirements asserted (all SATISFIED by default)
        state STANDBY;

        // PUMPING — discharge operations active; waterLevel elevated or rising
        // Linked: BPS-REQ-004 (DischargeCapacityRequirement) under continuous evaluation
        state PUMPING;

        // ALERT — IEC 60945 alarm window active; audible/visual annunciation
        // Linked: BPS-REQ-003 (AlarmResponseRequirement)
        state ALERT;

        // FAILOVER_ACTIVE — Pump A unavailable; Pump B handling full discharge load
        // Linked: BPS-REQ-006 (FailoverSwitchTimingRequirement), SR-004
        state FAILOVER_ACTIVE;

        // CRITICAL_FAULT — controller or dual-pump failure; SOLAS emergency response
        // Linked: FM-C-001 (RPN=140); manual intervention mandatory
        state CRITICAL_FAULT;

        // Startup
        entry;
        then STANDBY;

        // STANDBY → PUMPING: water level crosses trigger threshold
        // Sub-state: controller transitions MONITORING → PUMP_A_ACTIVE
        // guard: sys.sensor.waterLevel >= sys.controller.triggerLevel_m
        transition STANDBY_to_PUMPING first STANDBY then PUMPING;

        // PUMPING → ALERT: alarm condition detected by controller
        // Sub-state: controller transitions PUMP_A_ACTIVE → ALARM_TRIGGERED
        // guard: sys.alarm.isActive == true
        transition PUMPING_to_ALERT first PUMPING then ALERT;

        // PUMPING → FAILOVER_ACTIVE: Pump A fault detected
        // Sub-state: controller transitions PUMP_A_ACTIVE → FAILOVER
        // guard: sys.pumpA.flowRate == 0.0
        transition PUMPING_to_FAILOVER_ACTIVE first PUMPING then FAILOVER_ACTIVE;

        // PUMPING → CRITICAL_FAULT: controller hang (FM-C-001)
        // Sub-state: controller transitions PUMP_A_ACTIVE → FAULT
        // guard: sys.controller.responseTime_s > 5.0
        transition PUMPING_to_CRITICAL_FAULT first PUMPING then CRITICAL_FAULT;

        // PUMPING → STANDBY: dewatering complete
        // Sub-state: controller transitions PUMP_A_ACTIVE or DUAL_PUMP_ACTIVE → MONITORING
        // guard: sys.sensor.waterLevel < sys.controller.triggerLevel_m
        transition PUMPING_to_STANDBY first PUMPING then STANDBY;

        // FAILOVER_ACTIVE → PUMPING: Pump B reaches nominal flow (handoff complete)
        // Sub-state: controller transitions FAILOVER → DUAL_PUMP_ACTIVE
        // guard: sys.pumpB.flowRate > 0.0
        transition FAILOVER_ACTIVE_to_PUMPING first FAILOVER_ACTIVE then PUMPING;

        // ALERT → STANDBY: operator acknowledges alarm via HMI
        // Sub-state: controller transitions ALARM_TRIGGERED → MONITORING
        // guard: sys.ui.overrideActive == true
        transition ALERT_to_STANDBY first ALERT then STANDBY;

        // CRITICAL_FAULT → STANDBY: operator hardware reset
        // Sub-state: controller transitions FAULT → IDLE
        // guard: sys.ui.overrideActive == true (hardware reset)
        transition CRITICAL_FAULT_to_STANDBY first CRITICAL_FAULT then STANDBY;
    }
} // package BilgePump_StateMachine


Package BilgePump_StateMachine (f85b1475-a820-41ef-8c99-fc2b635eb1ca)


In [8]:
// model version: dd31677
package BilgePump {
    private import ScalarValues::*;
    public import BilgePump_Library::*;
    public import BilgePump_Architecture::*;
    public import BilgePump_Requirements::*;
    public import BilgePump_Analysis::*;
    public import BilgePump_Safety::*;
    public import BilgePump_FMEA::*;
    public import BilgePump_StateMachine::*;
}

Package BilgePump (f1a0c884-6ba5-49d8-819c-445321e9b639)


In [9]:
%repo http://localhost:9000

http://localhost:9000


In [10]:
%publish BilgePump

API base path: http://localhost:9000


Processing

.

.

.

...

.

.

Posting commit. Additions: 5371, Deletions: 5365, Modified elements: 5


Successfully posted commit: 4e7d0618-fd5c-4046-86c3-c1785f611edf on branch: (main) d1d8593b-b1a5-4b05-98ef-abc3a34b5aab


Saved to Project BilgePump (c59534ac-d59a-4ee6-9069-63a54c8aaaf2)


In [11]:
%eval BilgePump_Analysis::bpVerification.BPS_REQ_001_waterLevel

LiteralBoolean true (505855e1-2500-47a0-90d7-89e9d534ec20)


In [12]:
%eval BilgePump_Analysis::bpVerification.BPS_REQ_002_redundancy

LiteralBoolean true (18734619-fc7a-400f-af8a-2426155e95be)


In [13]:
%eval BilgePump_Analysis::bpVerification.BPS_REQ_003_alarm

LiteralBoolean true (f841d292-e70b-4ea9-b5b9-2941d8775e33)


In [14]:
%eval BilgePump_Analysis::bpVerification.BPS_REQ_004_discharge

LiteralBoolean true (c5155b8c-1e9b-47d4-bdb0-f557bb3b20fe)


In [15]:
%eval BilgePump_Analysis::bpVerification.physicsCheck

LiteralBoolean true (10074588-0403-4eaf-8e33-c952ce6e6c83)


In [16]:
%eval BilgePump_Analysis::bpTimingVerification.BPS_REQ_005_activation

LiteralBoolean true (a19fea4e-4490-4d92-82d5-1652ed8759dc)


In [17]:
%eval BilgePump_Analysis::bpTimingVerification.BPS_REQ_006_failover

LiteralBoolean true (bfda2a50-d02d-41f9-a410-44fed353ef2f)


In [18]:
%eval BilgePump_Analysis::bpTimingVerification.BPS_REQ_003_alarmTiming

LiteralBoolean true (84014a77-ddda-424b-b5ab-8c67f56052fd)


In [19]:
%eval BilgePump_Analysis::bpTimingVerification.timingBudgetOk

LiteralBoolean true (753eae8c-3915-45a0-8c36-4c56e0568f95)


In [20]:
%eval BilgePump_Analysis::bpFaultToleranceVerification.BPS_FT_001_sensorAccuracy

LiteralBoolean true (a9040c09-26a4-4452-a1c5-df4b563af91c)


In [21]:
%eval BilgePump_Analysis::bpFaultToleranceVerification.BPS_FT_002_effDischarge

LiteralBoolean true (3d69838d-ab13-434b-8055-f8ab9e289a45)


In [22]:
%eval BilgePump_Analysis::bpFaultToleranceVerification.BPS_FT_003_triggerAccuracy

LiteralBoolean true (ccbb645e-6064-43a6-b19c-ada8cbb3bfff)


In [23]:
%eval BilgePump_Analysis::bpFaultToleranceVerification.BPS_FT_004_responseWindow

LiteralBoolean true (44ac38fd-65e2-4381-92a7-f539c4f9c371)


In [24]:
%eval BilgePump_Analysis::bpFaultToleranceVerification.BPS_OOR_001_overrideOrdering

LiteralBoolean true (de179963-3d23-4cf1-b0ee-22f43d9d9d14)


In [25]:
%eval BilgePump_FMEA::FM_S_001_alarm

LiteralBoolean false (337e1e8c-abd8-4411-94d7-091f81698258)


In [26]:
%eval BilgePump_FMEA::FM_S_001_discharge

LiteralBoolean false (998ad772-6c90-44f7-a8af-cfb7b72d5669)


In [27]:
%eval BilgePump_FMEA::FM_PA_002_effDischarge

LiteralBoolean false (9283b79d-1d56-4115-a59f-1d5153a88b0b)


In [28]:
%eval BilgePump_FMEA::FM_PB_001_redundancy

LiteralBoolean false (59ef845d-cd43-4a80-82bf-cb0f8fb5be7a)


In [29]:
%eval BilgePump_FMEA::FM_C_001_activation

LiteralBoolean false (7c149910-d64e-4cf8-9d4d-283f6ad43a12)


In [30]:
%eval BilgePump_FMEA::FM_C_001_alarm

LiteralBoolean false (5dbef701-859a-47bf-bd80-c84b4cb43612)


In [31]:
%eval BilgePump_FMEA::FM_C_001_discharge

LiteralBoolean false (7c7aa81d-2630-4ed8-8767-50e09457f720)


In [32]:
%eval BilgePump_FMEA::FM_C_003_failover

LiteralBoolean false (e6c6879d-7792-49ad-b02d-8eb6f8207de7)


In [33]:
%eval BilgePump_FMEA::FM_C_003_discharge

LiteralBoolean false (028ea5c8-bfa7-4db0-b09c-33878cc039d4)


In [34]:
%eval BilgePump_Analysis::UQ_MARGIN_PROBE

LiteralBoolean true (f7d673ed-1211-4199-b5bf-5f0f2e13eabc)


In [35]:
%eval BilgePump_Analysis::bpSimEvidence.SIM_LEVEL_001

LiteralBoolean true (7950f8e1-3839-4cef-a1ac-1f0f64260821)


In [36]:
%eval BilgePump_Analysis::bpSimEvidence.SIM_RESP_005

LiteralBoolean true (ba342687-14ea-4efb-9497-7df68cfc3286)


In [37]:
%eval BilgePump_Analysis::bpSimEvidence.SIM_FO_006

LiteralBoolean true (f42d3d8e-2c78-41b4-8abf-8ea957a1407b)


In [38]:
%eval BilgePump_Analysis::bpSimEvidence.SIM_LEVEL_001_NEG

LiteralBoolean false (fdcc8f10-d9c4-426a-8576-9128d569d76e)
